# 主力片段与全合约价格的复权

本方法以每日主力映射确定连续主力片段，在分钟网格上构造共同的 log 价格偏移，再将该偏移应用于同一时点全部真实合约的 OHLC。首尾两条主力分钟保持调整前价格，累计换月差值在全部相邻实际分钟间隔之间等额分摊，成交量、成交额与持仓量保留原值。

本页给出日级分段、分钟与 log 输入准备、共同偏移复权、结果保存，使用 RB 正式行情展示计算、换约诊断和数学验收。全合约复权长表保存为同目录 `b02_MainContinuousAdjustment_demo.parquet`；下游用 `is_main` 选取主力分钟，用 `segment_id` 定位片段。

## 1. 输入与时间范围

复权区间就是用来计算复权偏移的交易日范围。本演示选取 2023-10-25 至 2025-10-20，两端包含，只用这段日期内的主力分钟行情计算偏移。

每日主力身份来自上游 [b01_MainContractSelection_demo.parquet](../b01_MainContractSelection/b01_MainContractSelection_demo.parquet)，先核对 Schema、生成代码和实际内容摘要。分钟行情与 Session 覆盖来自正式 silver 的 `fact_futures_minute`、`dim_futures_bar_calendar`，契约使用 `config/data_contracts.py`，湖根使用 `settings.futures_lake_root`。

| 范围 | 起点 | 终点 | 端点 |
|---|---|---|---|
| 每日映射读取区间 | 2015-01-01 | 2026-09-01 | 两端包含 |
| 分钟行情读取交易日区间 | 2023-10-25 | 2025-10-20 | 两端包含 |
| 复权交易日区间 | 2023-10-25 | 2025-10-20 | 两端包含 |

时区为 `Asia/Shanghai`。分钟归属直接使用 silver 的 `trading_date`；夜盘可落在前一个自然日，不用自然日期或固定钟点推断。读取前按交易所、品种、年、月裁剪，再按交易日过滤。实际分钟首末 `bar_at` 在 demo 中登记。首尾指实际主力分钟序列的第一条与最后一条；全合约在这两个时间键共享零偏移。复权范围决定价格锚点和分摊间隔数，改变范围需重新计算。

信任 clean silver 已提交的主键及业务质量证明；读取仅核对物理字段、类型和 metadata，并检查本方法关联、分钟边界、正价格等计算前提。

In [1]:
import hashlib
import json
import pathlib
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "04_Research"))
        break
else:
    raise RuntimeError("未找到项目根目录")

print(sys.executable)
import pyarrow.parquet as pq
from IPython.display import display
from a01_Methods.b01_MainContractSelection.b01_MainContractSelection import (
    MAIN_CONTRACT_DAILY_SCHEMA,
)


E:\anaconda3\envs\latitude_env_v2\python.exe


## 2. 连续片段规则

每日映射提供完整交易日网格，按交易日升序处理。在同一个交易所和品种内，相邻交易日主力合约相同，属于同一片段；主力变化时开启新片段。周末和节假日不按自然日间隔切断片段。

未识别日保留记录，`segment_id` 为空，并切断前后已识别片段。同一合约离开主力后再次出现，得到新的片段编号。编号在本次复权区间内按时间顺序从 1 开始，不以合约代码排序或合并。

每日片段表保留交易日、合约、片段身份及片段起止交易日；片段汇总表每个片段一行。`is_final_segment=True` 表示这一行对应选定日期范围内、按时间最后的一段主力行情。本演示的最后片段是 RB2601.XSGE，覆盖交易日 2025-09-01 至 2025-10-20；该片段也参与间隔均摊，只有序列首尾固定为原价。

完整复权还要求区间首尾具备主力身份、区间内无未识别日；分段函数保留缺口供核查，复权函数拒绝不满足这些条件的输入。

## 3. log 域共同偏移

对正的有限价格先取自然对数：\(x_{c,f,i}=\ln P_{c,f,i}\)。\(c\) 表示真实合约，\(f\) 为 open、high、low、close，\(i\) 为实际主力分钟网格位置。缺失价格保留为空；非空价格不满足正值和有限性时停止复权计算。

主力片段按时间编号为 \(1,\ldots,K\)。从片段 \(k\) 切换至 \(k+1\) 的 log 缺口为：

\[
g_k=\ln O^{\mathrm{first}}_{k+1}-\ln C^{\mathrm{last}}_k.
\]

锚点取前一片段的最后收盘和后一片段的首个开盘，分别属于这两个片段的真实分钟行情。锚点缺失时停止计算。夜盘归属与片段分钟边界由正式交易日字段确定。

对片段 \(k\)，累计后续缺口 \(B_k=\sum_{j=k}^{K-1}g_j\)，最后片段 \(B_K=0\)。记累计换月差值 \(G=B_1\)，实际主力分钟共 \(N\) 条，位置 \(i=0,\ldots,N-1\)。当 \(N\ge2\) 时，将 \(G\) 等额分摊到全部 \(N-1\) 个相邻间隔，每个间隔分摊 \(q=G/(N-1)\)；换月衔接和最后片段内的间隔都参与，按行序等分，不按经过的自然时间加权。

构造从 \(G\) 线性变化到 0 的线性回扣 \(D_i\)，以累计偏移减去它：

\[
D_i=G\left(1-\frac{i}{N-1}\right),\qquad
a_i=B_{k(i)}-D_i,
\]
\[
x^{\mathrm{adj}}_{c,f,i}=x_{c,f,i}+a_i,\qquad
P^{\mathrm{adj}}_{c,f,i}=\exp(x^{\mathrm{adj}}_{c,f,i}).
\]

首行 \(a_0=G-G=0\)，末行 \(a_{N-1}=0-0=0\)，因此首尾保持原价。单片段时 \(G=0\) 且全部偏移为零；只有一条分钟时没有间隔，直接取零偏移。实现时零偏移位置直接保留原始 OHLC 数值，避免 log/exp 往返的浮点舍入改变锚定价格。

同一分钟全部合约、全部 OHLC 字段共用一个 \(a_i\)，因此同一时点的 log 价差保持不变。数量字段保留原值，不取 log、不参与偏移。普通相邻分钟的 log 价格变化增加 \(q\)；换月处先减去原始 log 缺口，再增加同样的 \(q\)，所以换约后首开减换约前末收等于 \(q\)。这一数值是全区间统一的分摊量。本演示 \(N=164385\)，共分摊到 164384 个间隔。

分钟缺失保留为诊断，不补价格、不改主力、不补偏移。日历 Session 的期望网格为 `(session_start_at, session_end_at]`，仅为主力构造该网格并与实际分钟对照。区间内缺失不另改主力片段编号；线性分摊仅按实际主力分钟行计数。缺失若位于某片段首条之前或末条之后、存在无主力身份日、整段无行情，均停止复权，避免用后续价格替代锚点。同一主力时点只能有一行，关联产生的重复键或时间重叠硬失败。

各合约的分钟行情可以不完全对应。例如，10:01 其他合约有行情，但主力合约没有行情，就取不到该分钟的复权偏移。

这些行情仍保留原价和 log 价；偏移、复权后 log 价和复权后价格留空，标记为 `no_main_offset`。这些时间点单独列出，不借用前后分钟的偏移。

In [2]:
from __future__ import annotations

from datetime import date
import numpy as np
import pandas as pd
import pyarrow as pa

from config.data_contracts import FUTURES_MINUTE_SCHEMA

MAIN_CONTINUOUS_ADJUSTMENT_VERSION = "0.4.0"
ADJUSTMENT_TIMEZONE = "Asia/Shanghai"
MAIN_SEGMENT_DAILY_SCHEMA = pa.schema([
    pa.field("exchange_code", pa.string(), nullable=False),
    pa.field("underlying_code", pa.string(), nullable=False),
    pa.field("trading_date", pa.date32(), nullable=False),
    pa.field("signal_trading_date", pa.date32()),
    pa.field("main_contract_code", pa.string()),
    pa.field("segment_id", pa.int32()),
    pa.field("segment_start_date", pa.date32()),
    pa.field("segment_end_date", pa.date32()),
    pa.field("selection_status", pa.string(), nullable=False),
    pa.field("is_roll", pa.bool_()),
], metadata={
    b"method_name": b"main_continuous_adjustment_segmentation",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"grain_zh": "每个目标交易日一行，未识别日的片段编号为空".encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",
})
MAIN_SEGMENT_SUMMARY_SCHEMA = pa.schema([
    pa.field("exchange_code", pa.string(), nullable=False),
    pa.field("underlying_code", pa.string(), nullable=False),
    pa.field("segment_id", pa.int32(), nullable=False),
    pa.field("main_contract_code", pa.string(), nullable=False),
    pa.field("segment_start_date", pa.date32(), nullable=False),
    pa.field("segment_end_date", pa.date32(), nullable=False),
    pa.field("trading_day_count", pa.int32(), nullable=False),
    pa.field("is_final_segment", pa.bool_(), nullable=False),
], metadata={
    b"method_name": b"main_continuous_adjustment_segmentation",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"grain_zh": "每个连续已识别主力片段一行".encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",
})

OHLC_FIELDS = ("open", "high", "low", "close")
QUANTITY_FIELDS = ("volume", "money", "open_interest")
ALL_CONTRACT_LOG_MINUTE_SCHEMA = pa.schema([
    *(pa.field(f"raw_{field.name}", field.type, nullable=field.nullable)
      if field.name in OHLC_FIELDS else field for field in FUTURES_MINUTE_SCHEMA),
    *(pa.field(f"log_{name}", pa.float64()) for name in OHLC_FIELDS),
    pa.field("main_contract_code", pa.string()),
    pa.field("segment_id", pa.int32()),
    pa.field("is_main", pa.bool_(), nullable=False),
], metadata={
    b"method_name": b"main_continuous_adjustment_log_input",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",
    b"price_domain": b"raw_* original; log_* natural logarithm",
})
MINUTE_LOG_OFFSET_SCHEMA = pa.schema([
    FUTURES_MINUTE_SCHEMA.field("trading_date"),
    FUTURES_MINUTE_SCHEMA.field("bar_at"),
    pa.field("main_contract_code", pa.string(), nullable=False),
    pa.field("segment_id", pa.int32(), nullable=False),
    pa.field("backward_log_shift", pa.float64(), nullable=False),
    pa.field("linear_log_deduction", pa.float64(), nullable=False),
    pa.field("log_adjustment", pa.float64(), nullable=False),
], metadata={
    b"method_name": b"main_continuous_adjustment_offset",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
})

ALL_CONTRACT_ADJUSTED_MINUTE_SCHEMA = pa.schema([
    *ALL_CONTRACT_LOG_MINUTE_SCHEMA,
    pa.field("backward_log_shift", pa.float64()),
    pa.field("linear_log_deduction", pa.float64()),
    pa.field("log_adjustment", pa.float64()),
    pa.field("adjustment_status", pa.string(), nullable=False),
    *(pa.field(f"adjusted_log_{name}", pa.float64()) for name in OHLC_FIELDS),
    *(pa.field(f"adjusted_{name}", pa.float64()) for name in OHLC_FIELDS),
], metadata={
    b"method_name": b"main_continuous_adjustment",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"grain_zh": "每个真实合约的一分钟行情一行".encode(),
    b"primary_key": b"contract_code,bar_at",
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",
    b"price_domains": b"raw_* original; log_* natural log; adjusted_log_* adjusted natural log; adjusted_* price",
    b"offset_rule": b"g=new first log_open-old last log_close; B=reverse cumulative g; D=linspace(G,0,N) over all observed main minutes; A=B-D; N-1 intervals including rolls; first/last A=0",
    b"missing_offset_rule": b"no_main_offset: adjusted values null; raw/log/quantities preserved; no filling",
})


In [3]:
def build_main_contract_segments(
    main_contract_daily_df: pd.DataFrame,
    *,
    exchange_code: str,
    underlying_code: str,
    adjustment_start_date: str | date | None = None,
    adjustment_end_date: str | date | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """按连续交易日划分主力片段，保留未识别日。

    Args:
        main_contract_daily_df: 已核对的完整每日映射，日期采用局部 Schema 类型。
        exchange_code: 目标交易所代码，如 "XSGE"。
        underlying_code: 目标品种代码，如 "RB"。
        adjustment_start_date: 复权区间起点，含当日；None 使用映射首日。
        adjustment_end_date: 复权区间终点，含当日；None 使用映射末日。

    Returns:
        (main_segment_daily_df, main_segment_summary_df)：每日片段表与片段汇总表。
    """
    # 1. 先固定复权区间；片段编号和最后片段均由该区间定义。
    main_segment_daily_df = main_contract_daily_df.loc[
        (main_contract_daily_df["exchange_code"] == exchange_code)
        & (main_contract_daily_df["underlying_code"] == underlying_code),
        ["exchange_code", "underlying_code", "trading_date", "signal_trading_date",
         "main_contract_code", "selection_status", "is_roll"],
    ].sort_values("trading_date").reset_index(drop=True)
    if main_segment_daily_df.empty:
        raise ValueError("目标品种没有每日主力映射")
    adjustment_start_date = (
        pd.Timestamp(adjustment_start_date).date() if adjustment_start_date is not None
        else main_segment_daily_df["trading_date"].min()
    )
    adjustment_end_date = (
        pd.Timestamp(adjustment_end_date).date() if adjustment_end_date is not None
        else main_segment_daily_df["trading_date"].max()
    )
    if adjustment_start_date > adjustment_end_date:
        raise ValueError("复权区间起点不能晚于终点")
    main_segment_daily_df = main_segment_daily_df.loc[
        main_segment_daily_df["trading_date"].between(adjustment_start_date, adjustment_end_date)
    ].reset_index(drop=True)
    if main_segment_daily_df.empty:
        raise ValueError("复权区间内没有每日主力映射")

    # 2. 合约变化或前一日未识别时开启新片段；空主力日不获得片段编号。
    identified_main_mask = main_segment_daily_df["main_contract_code"].notna()
    segment_start_mask = identified_main_mask & main_segment_daily_df["main_contract_code"].ne(
        main_segment_daily_df["main_contract_code"].shift()
    ).fillna(True)
    # 起点标记先转整数，再累计为片段编号。
    main_segment_daily_df["segment_id"] = segment_start_mask.astype("int32").cumsum().where(identified_main_mask).astype("Int32")

    # 3. 按片段编号汇总，不能按合约代码合并重复出现的片段。
    main_segment_summary_df = main_segment_daily_df.loc[identified_main_mask].groupby(
        "segment_id", as_index=False, sort=True,
    ).agg(
        exchange_code=("exchange_code", "first"),
        underlying_code=("underlying_code", "first"),
        main_contract_code=("main_contract_code", "first"),
        segment_start_date=("trading_date", "min"),
        segment_end_date=("trading_date", "max"),
        trading_day_count=("trading_date", "size"),
    )
    main_segment_summary_df["is_final_segment"] = (
        main_segment_summary_df["segment_id"].eq(main_segment_summary_df["segment_id"].max())
    )
    main_segment_daily_df = main_segment_daily_df.merge(
        main_segment_summary_df[["segment_id", "segment_start_date", "segment_end_date"]],
        on="segment_id", how="left", validate="many_to_one",
    )

    # 4. 两张结果按本方法局部 Schema 转换，保留日期、可空编号与布尔类型。
    main_segment_daily_table = pa.Table.from_pandas(
        main_segment_daily_df, schema=MAIN_SEGMENT_DAILY_SCHEMA, preserve_index=False,
    ).replace_schema_metadata(MAIN_SEGMENT_DAILY_SCHEMA.metadata)
    main_segment_summary_table = pa.Table.from_pandas(
        main_segment_summary_df, schema=MAIN_SEGMENT_SUMMARY_SCHEMA, preserve_index=False,
    ).replace_schema_metadata(MAIN_SEGMENT_SUMMARY_SCHEMA.metadata)
    return (
        main_segment_daily_table.to_pandas(types_mapper=pd.ArrowDtype),
        main_segment_summary_table.to_pandas(types_mapper=pd.ArrowDtype),
    )


In [4]:
def prepare_main_contract_minutes(
    futures_minute_df: pd.DataFrame,
    main_segment_daily_df: pd.DataFrame,
    bar_calendar_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """关联主力片段并取 log，保留原价、数量与缺失证据。

    Args:
        futures_minute_df: 按正式契约读取的目标品种全合约分钟行情。
        main_segment_daily_df: 同一范围、单一品种的每日主力片段表。
        bar_calendar_df: 同范围的正式 1m Session 日历，含期望和实测条数。

    Returns:
        全合约 log 长表、主力 log 分钟表、覆盖及缺失诊断字典。
    """
    daily_df = main_segment_daily_df.sort_values("trading_date").reset_index(drop=True)
    if daily_df.empty or daily_df["trading_date"].duplicated().any():
        raise ValueError("每日主力片段为空或交易日重复")
    identities_df = daily_df[["exchange_code", "underlying_code"]].drop_duplicates()
    if len(identities_df) != 1:
        raise ValueError("一次分钟准备只能处理一个交易所和品种")
    exchange_code, underlying_code = identities_df.iloc[0]
    start_date, end_date = daily_df["trading_date"].min(), daily_df["trading_date"].max()
    minute_df = futures_minute_df.loc[
        futures_minute_df["exchange_code"].eq(exchange_code)
        & futures_minute_df["underlying_code"].eq(underlying_code)
        & futures_minute_df["trading_date"].between(start_date, end_date)
    ].copy()
    if minute_df.empty:
        raise ValueError("复权区间没有分钟行情")
    if pa.Array.from_pandas(minute_df["bar_at"]).type != FUTURES_MINUTE_SCHEMA.field("bar_at").type:
        raise ValueError("bar_at 必须保留正式 Asia/Shanghai 时间戳类型")

    # 只按 silver trading_date 关联，many_to_one 防止日级映射放大分钟行数。
    all_contract_log_minute_df = minute_df.rename(
        columns={name: f"raw_{name}" for name in OHLC_FIELDS}
    ).merge(daily_df[["trading_date", "main_contract_code", "segment_id"]],
            on="trading_date", how="left", validate="many_to_one", indicator=True)
    if not all_contract_log_minute_df["_merge"].eq("both").all():
        raise ValueError("分钟交易日不在每日主力映射中")
    all_contract_log_minute_df = all_contract_log_minute_df.drop(columns="_merge")
    if len(all_contract_log_minute_df) != len(minute_df) or all_contract_log_minute_df.duplicated(["contract_code", "bar_at"]).any():
        raise ValueError("关联后的分钟行数改变或合约时间键重复")
    all_contract_log_minute_df["is_main"] = all_contract_log_minute_df["contract_code"].eq(
        all_contract_log_minute_df["main_contract_code"]
    ).fillna(False)
    for name in OHLC_FIELDS:
        prices = all_contract_log_minute_df[f"raw_{name}"].to_numpy(dtype=float, na_value=np.nan)
        present = all_contract_log_minute_df[f"raw_{name}"].notna().to_numpy(dtype=bool)
        invalid = present & ((prices <= 0) | ~np.isfinite(prices))
        if invalid.any():
            sample_df = all_contract_log_minute_df.loc[invalid, ["contract_code", "bar_at", f"raw_{name}"]].head(5)
            raise ValueError(f"{name} 含 {int(invalid.sum())} 个非正或非有限价格：\n{sample_df}")
        log_prices = np.full(len(prices), np.nan)
        log_prices[present] = np.log(prices[present])
        all_contract_log_minute_df[f"log_{name}"] = pd.array(log_prices, dtype=pd.ArrowDtype(pa.float64()))
    all_contract_log_minute_df = pa.Table.from_pandas(
        all_contract_log_minute_df, schema=ALL_CONTRACT_LOG_MINUTE_SCHEMA, preserve_index=False,
    ).to_pandas(types_mapper=pd.ArrowDtype).sort_values(["bar_at", "contract_code"]).reset_index(drop=True)
    main_log_minute_df = all_contract_log_minute_df.loc[all_contract_log_minute_df["is_main"]].reset_index(drop=True)
    if main_log_minute_df["bar_at"].duplicated().any():
        raise ValueError("主力分钟时间重叠，不能建立唯一共同偏移")

    # 日历已提交的覆盖证明直接展示；仅构造选中主力的期望网格定位本算法缺口。
    session_coverage_df = bar_calendar_df.loc[
        bar_calendar_df["bar_frequency"].eq("1m")
        & bar_calendar_df["exchange_code"].eq(exchange_code)
        & bar_calendar_df["underlying_code"].eq(underlying_code)
        & bar_calendar_df["trading_date"].between(start_date, end_date),
        ["contract_code", "trading_date", "session_number", "session_start_at", "session_end_at",
         "schedule_status", "schedule_signal_reason", "is_fetch_required", "is_fetch_completed",
         "expected_bar_count", "actual_bar_count", "missing_bar_count", "quality_status", "quality_reason"],
    ].merge(daily_df[["trading_date", "main_contract_code", "segment_id"]],
            on="trading_date", how="left", validate="many_to_one")
    session_coverage_df["is_main"] = session_coverage_df["contract_code"].eq(session_coverage_df["main_contract_code"]).fillna(False)
    main_sessions_df = session_coverage_df.loc[session_coverage_df["is_main"] & session_coverage_df["is_fetch_required"]]
    missing_calendar_dates = daily_df.loc[daily_df["main_contract_code"].notna() & ~daily_df["trading_date"].isin(main_sessions_df["trading_date"])]
    if not missing_calendar_dates.empty:
        raise ValueError(f"主力缺少 required Session 日历：{missing_calendar_dates['trading_date'].tolist()}")
    expected_parts = []
    for session in main_sessions_df.itertuples(index=False):
        if pd.isna(session.session_start_at) or pd.isna(session.session_end_at):
            raise ValueError("required 主力 Session 缺少起止时刻")
        timestamps = pd.date_range(session.session_start_at, session.session_end_at, freq="min", inclusive="right")
        if len(timestamps) != session.expected_bar_count:
            raise ValueError("主力 Session 起止时刻与期望分钟数不一致")
        expected_parts.append(pd.DataFrame({
            "trading_date": session.trading_date, "bar_at": timestamps,
            "contract_code": session.contract_code, "segment_id": session.segment_id,
            "session_number": session.session_number,
        }))
    if not expected_parts:
        raise ValueError("没有可识别主力的期望分钟网格")
    expected_main_minute_df = pd.concat(expected_parts, ignore_index=True)
    expected_main_minute_df["bar_at"] = pd.array(expected_main_minute_df["bar_at"], dtype=pd.ArrowDtype(FUTURES_MINUTE_SCHEMA.field("bar_at").type))
    expected_main_minute_df["trading_date"] = pd.array(expected_main_minute_df["trading_date"], dtype=pd.ArrowDtype(pa.date32()))
    if expected_main_minute_df.duplicated(["contract_code", "bar_at"]).any() or expected_main_minute_df["bar_at"].duplicated().any():
        raise ValueError("生成的主力期望分钟网格存在 Session 重叠")
    main_grid_comparison_df = expected_main_minute_df.merge(
        main_log_minute_df[["contract_code", "bar_at", "trading_date"]],
        on=["contract_code", "bar_at", "trading_date"], how="outer", validate="one_to_one", indicator=True,
    )
    if main_grid_comparison_df["_merge"].eq("right_only").any():
        raise ValueError("主力分钟落在本范围日历期望网格之外")
    missing_main_minute_df = main_grid_comparison_df.loc[main_grid_comparison_df["_merge"].eq("left_only")].drop(columns="_merge").sort_values("bar_at").reset_index(drop=True)
    main_daily_coverage_df = daily_df[["trading_date", "main_contract_code", "segment_id"]].copy()
    for column, source_df in (("expected_minute_count", expected_main_minute_df),
                              ("observed_minute_count", main_log_minute_df),
                              ("missing_minute_count", missing_main_minute_df)):
        counts = source_df.groupby("trading_date").size()
        main_daily_coverage_df[column] = main_daily_coverage_df["trading_date"].map(counts).fillna(0).astype(pd.ArrowDtype(pa.int32()))
    all_minute_times_df = all_contract_log_minute_df[["trading_date", "bar_at"]].drop_duplicates()
    unsupported_times_df = all_minute_times_df.merge(main_log_minute_df[["trading_date", "bar_at"]],
        on=["trading_date", "bar_at"], how="left", validate="one_to_one", indicator=True)
    unsupported_times_df = unsupported_times_df.loc[unsupported_times_df["_merge"].eq("left_only")].drop(columns="_merge")
    diagnostics = {
        "session_coverage_df": session_coverage_df,
        "missing_main_minute_df": missing_main_minute_df,
        "main_daily_coverage_df": main_daily_coverage_df,
        "unresolved_main_daily_df": daily_df.loc[daily_df["main_contract_code"].isna()],
        "all_contract_times_without_main_df": unsupported_times_df,
    }
    return all_contract_log_minute_df, main_log_minute_df, diagnostics


In [5]:
def adjust_main_contract_minutes(
    main_log_minute_df: pd.DataFrame,
    all_contract_log_minute_df: pd.DataFrame,
    main_segment_daily_df: pd.DataFrame,
    missing_main_minute_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """在 log 域按全部相邻间隔均摊累计换月差值，保持首尾原价。

    同一时间键的全合约 OHLC 共用偏移；间隔按实际主力分钟行序计数。

    Args:
        main_log_minute_df: prepare_main_contract_minutes 返回的主力 log 行情。
        all_contract_log_minute_df: 同次准备的全合约原价与 log 长表。
        main_segment_daily_df: 固定复权范围的日级片段，须全部识别。
        missing_main_minute_df: 同次准备的期望主力分钟缺失明细。

    Returns:
        复权主力、复权全合约、逐分钟偏移、片段/换约及缺口诊断。
    """
    main_df = main_log_minute_df.sort_values("bar_at").reset_index(drop=True)
    daily_df = main_segment_daily_df.sort_values("trading_date")
    if main_df.empty or daily_df.empty or daily_df["segment_id"].isna().any():
        raise ValueError("复权范围必须有行情且每日主力全部识别")
    if main_df["bar_at"].duplicated().any():
        raise ValueError("主力分钟时间重叠")
    segment_ids = daily_df["segment_id"].drop_duplicates().tolist()
    if main_df["segment_id"].drop_duplicates().tolist() != segment_ids:
        raise ValueError("片段整段缺行情或分钟片段时间顺序错误")
    selected_all_df = all_contract_log_minute_df.loc[all_contract_log_minute_df["is_main"]].sort_values("bar_at").reset_index(drop=True)
    if not main_df.equals(selected_all_df):
        raise ValueError("主力与全合约输入不是同一次分钟准备结果")

    segment_diagnostics_df = main_df.groupby("segment_id", sort=False).agg(
        main_contract_code=("contract_code", "first"),
        first_bar_at=("bar_at", "first"), last_bar_at=("bar_at", "last"),
        first_trading_date=("trading_date", "first"), last_trading_date=("trading_date", "last"),
        minute_count=("bar_at", "size"),
    ).reset_index()
    # 缺失发生在片段边缘时拒绝计算，不能把后续有效行情冒充首末锚点。
    if not missing_main_minute_df.empty:
        missing_bounds_df = missing_main_minute_df.merge(segment_diagnostics_df[["segment_id", "first_bar_at", "last_bar_at"]], on="segment_id", validate="many_to_one")
        bad_boundary = (missing_bounds_df["bar_at"] < missing_bounds_df["first_bar_at"]) | (missing_bounds_df["bar_at"] > missing_bounds_df["last_bar_at"])
        if bad_boundary.any():
            raise ValueError(f"片段首末边界缺失 {int(bad_boundary.sum())} 分钟，拒绝替换锚点")
    first_rows_df = main_df.drop_duplicates("segment_id", keep="first").reset_index(drop=True)
    last_rows_df = main_df.drop_duplicates("segment_id", keep="last").reset_index(drop=True)
    old_log_close = last_rows_df["log_close"].to_numpy(dtype=float, na_value=np.nan)[:-1]
    new_log_open = first_rows_df["log_open"].to_numpy(dtype=float, na_value=np.nan)[1:]
    if not np.isfinite(old_log_close).all() or not np.isfinite(new_log_open).all():
        raise ValueError("换约首开或末收 log 锚点缺失或非有限")

    # g>0 表示新主力开盘更高；旧片段加上后续 g 的总和，以末片段为后复权锚点。
    gaps = new_log_open - old_log_close
    backward_shifts = np.r_[np.cumsum(gaps[::-1])[::-1], 0.0]
    segment_diagnostics_df["backward_log_shift"] = pd.array(backward_shifts, dtype=pd.ArrowDtype(pa.float64()))
    total_log_gap = float(backward_shifts[0])
    final_segment_id = segment_ids[-1]
    main_minute_count = len(main_df)
    interval_count = main_minute_count - 1
    per_interval_log_gap = total_log_gap / interval_count if interval_count else 0.0
    # D 覆盖全部实际主力分钟；相邻 D 相差 G/(N-1)，换月间隔同样参与。
    # N=1 时必为单片段，G=0，得到唯一的零偏移。
    deductions = np.linspace(total_log_gap, 0.0, main_minute_count)
    shift_by_segment = dict(zip(segment_ids, backward_shifts))
    offsets = main_df["segment_id"].map(shift_by_segment).to_numpy(dtype=float) - deductions
    minute_log_offset_df = main_df[["trading_date", "bar_at", "contract_code", "segment_id"]].rename(columns={"contract_code": "main_contract_code"}).copy()
    minute_log_offset_df["backward_log_shift"] = main_df["segment_id"].map(shift_by_segment)
    minute_log_offset_df["linear_log_deduction"] = deductions
    minute_log_offset_df["log_adjustment"] = offsets
    minute_log_offset_df = pa.Table.from_pandas(minute_log_offset_df, schema=MINUTE_LOG_OFFSET_SCHEMA, preserve_index=False).to_pandas(types_mapper=pd.ArrowDtype)

    # 以 trading_date + bar_at 广播；无主力偏移的合约时点保留空值，绝不补偏移。
    broadcast_columns = ["trading_date", "bar_at", "backward_log_shift", "linear_log_deduction", "log_adjustment"]
    adjusted_all_contract_minute_df = all_contract_log_minute_df.merge(
        minute_log_offset_df[broadcast_columns], on=["trading_date", "bar_at"],
        how="left", validate="many_to_one",
    )
    if len(adjusted_all_contract_minute_df) != len(all_contract_log_minute_df) or adjusted_all_contract_minute_df.duplicated(["contract_code", "bar_at"]).any():
        raise ValueError("偏移广播引入重复时间键或改变行数")
    broadcast_offsets = adjusted_all_contract_minute_df["log_adjustment"].to_numpy(dtype=float, na_value=np.nan)
    adjusted_all_contract_minute_df["adjustment_status"] = pd.array(
        np.where(np.isfinite(broadcast_offsets), "adjusted", "no_main_offset"), dtype=pd.ArrowDtype(pa.string()))
    zero_offset = broadcast_offsets == 0
    for name in OHLC_FIELDS:
        original_log = adjusted_all_contract_minute_df[f"log_{name}"].to_numpy(dtype=float, na_value=np.nan)
        adjusted_log = original_log + broadcast_offsets
        with np.errstate(over="raise", invalid="raise"):
            adjusted_prices = np.exp(adjusted_log)
        # A=0 时直接复制原价，保证首尾两条分钟的原价逐值不变。
        original_prices = adjusted_all_contract_minute_df[f"raw_{name}"].to_numpy(dtype=float, na_value=np.nan)
        adjusted_prices[zero_offset] = original_prices[zero_offset]
        if ((~np.isnan(adjusted_prices)) & (~np.isfinite(adjusted_prices) | (adjusted_prices <= 0))).any():
            raise ValueError("复权后价格发生数值溢出或下溢")
        adjusted_all_contract_minute_df[f"adjusted_log_{name}"] = pd.array(adjusted_log, dtype=pd.ArrowDtype(pa.float64()))
        adjusted_all_contract_minute_df[f"adjusted_{name}"] = pd.array(adjusted_prices, dtype=pd.ArrowDtype(pa.float64()))
    adjusted_all_contract_minute_df = pa.Table.from_pandas(
        adjusted_all_contract_minute_df, schema=ALL_CONTRACT_ADJUSTED_MINUTE_SCHEMA, preserve_index=False,
    ).to_pandas(types_mapper=pd.ArrowDtype)
    adjusted_main_minute_df = adjusted_all_contract_minute_df.loc[adjusted_all_contract_minute_df["is_main"]].sort_values("bar_at").reset_index(drop=True)
    first_positions = np.flatnonzero(main_df["segment_id"].ne(main_df["segment_id"].shift()).fillna(True).to_numpy(dtype=bool))
    new_positions = first_positions[1:]
    old_positions = new_positions - 1
    roll_diagnostics_df = pd.DataFrame({
        "old_segment_id": segment_ids[:-1], "new_segment_id": segment_ids[1:],
        "old_contract_code": last_rows_df["contract_code"].tolist()[:-1],
        "new_contract_code": first_rows_df["contract_code"].tolist()[1:],
        "old_bar_at": last_rows_df["bar_at"].tolist()[:-1],
        "new_bar_at": first_rows_df["bar_at"].tolist()[1:],
        "gap_log": gaps, "old_linear_log_deduction": deductions[old_positions],
        "new_linear_log_deduction": deductions[new_positions],
        "expected_residual_log": np.full(len(gaps), per_interval_log_gap),
        "actual_residual_log": adjusted_main_minute_df["adjusted_log_open"].to_numpy(dtype=float, na_value=np.nan)[new_positions]
            - adjusted_main_minute_df["adjusted_log_close"].to_numpy(dtype=float, na_value=np.nan)[old_positions],
    })
    assert offsets[0] == 0 and offsets[-1] == 0
    assert np.allclose(roll_diagnostics_df["actual_residual_log"], roll_diagnostics_df["expected_residual_log"], atol=2e-14, rtol=1e-10)
    segment_diagnostics_df["missing_minute_count"] = segment_diagnostics_df["segment_id"].map(missing_main_minute_df.groupby("segment_id").size()).fillna(0).astype(pd.ArrowDtype(pa.int32()))
    segment_diagnostics_df["is_final_segment"] = segment_diagnostics_df["segment_id"].eq(final_segment_id)
    diagnostics = {
        "segment_diagnostics_df": segment_diagnostics_df,
        "roll_diagnostics_df": roll_diagnostics_df,
        "all_contract_times_without_offset_df": adjusted_all_contract_minute_df.loc[
            adjusted_all_contract_minute_df["log_adjustment"].isna(), ["trading_date", "bar_at"]].drop_duplicates(),
        "missing_main_minute_df": missing_main_minute_df,
        "main_minute_count": main_minute_count,
        "interval_count": interval_count,
        "per_interval_log_gap": per_interval_log_gap,
        "total_log_gap": total_log_gap,
    }
    return adjusted_main_minute_df, adjusted_all_contract_minute_df, minute_log_offset_df, diagnostics


In [6]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os


def arrow_content_sha256(table: pa.Table, schema: pa.Schema, sort_keys: list[tuple[str, str]]) -> str:
    """对明确 Schema、规范行序和合并 chunks 后的 Arrow IPC 内容取摘要。

    Args:
        table: 待核对的实际数据。
        schema: 不含 demo_identity 的权威输入或局部输出 Schema。
        sort_keys: 稳定行序，如 [("bar_at", "ascending"), ("contract_code", "ascending")]。
    """
    canonical_table = table.select(schema.names).cast(schema, safe=True).replace_schema_metadata(schema.metadata)
    canonical_table = canonical_table.combine_chunks().sort_by(sort_keys).combine_chunks()
    content_buffer = pa.BufferOutputStream()
    with pa.ipc.new_stream(content_buffer, canonical_table.schema) as writer:
        writer.write_table(canonical_table)
    return hashlib.sha256(content_buffer.getvalue().to_pybytes()).hexdigest()


def save_main_adjustment_demo(
    adjusted_all_contract_minute_df: pd.DataFrame,
    *,
    demo_path: str | Path,
    demo_identity: dict,
) -> dict:
    """保存单个全合约 demo，回读验收；身份与内容相同时保留已有字节。

    Args:
        adjusted_all_contract_minute_df: 复权函数返回的完整全合约长表。
        demo_path: 本方法的 *_demo.parquet 路径。
        demo_identity: 固定范围、实际输入、方法依赖、参数及运行环境身份。

    Returns:
        已验收身份、文件大小及是否保留原文件。
    """
    import pyarrow.parquet as pq

    demo_path = Path(demo_path)
    schema = ALL_CONTRACT_ADJUSTED_MINUTE_SCHEMA
    output_table = pa.Table.from_pandas(adjusted_all_contract_minute_df, schema=schema, preserve_index=False).replace_schema_metadata(schema.metadata)
    output_sort_keys = [("bar_at", "ascending"), ("contract_code", "ascending")]
    output_table = output_table.sort_by(output_sort_keys).combine_chunks()
    for field in schema:
        if not field.nullable and output_table[field.name].null_count:
            raise ValueError(f"输出非空字段 {field.name} 出现空值")
    expected_identity = {**demo_identity,
        "output_rows": output_table.num_rows,
        "output_schema_sha256": hashlib.sha256(schema.serialize().to_pybytes()).hexdigest(),
        "output_content_sha256": arrow_content_sha256(output_table, schema, output_sort_keys),
        "output_sort_keys": output_sort_keys,
    }
    # JSON 往返统一 tuple/list 等表示，避免同一次身份在回读后产生形式差异。
    expected_identity = json.loads(json.dumps(expected_identity, ensure_ascii=False, allow_nan=False))
    preserved_existing = False
    if demo_path.exists():
        existing_table = pq.read_table(demo_path)
        existing_metadata = existing_table.schema.metadata or {}
        if not existing_table.schema.equals(schema, check_metadata=False) or b"demo_identity" not in existing_metadata:
            raise RuntimeError("已有 demo Schema 或身份不兼容，请先核查已有文件")
        if existing_metadata.get(b"method_name") != schema.metadata[b"method_name"]:
            raise RuntimeError("已有 demo 方法身份不兼容")
        existing_identity = json.loads(existing_metadata[b"demo_identity"])
        existing_schema = existing_table.schema.with_metadata({key: value for key, value in existing_metadata.items() if key != b"demo_identity"})
        existing_content_sha256 = arrow_content_sha256(existing_table, existing_schema, output_sort_keys)
        if existing_content_sha256 != existing_identity["output_content_sha256"]:
            raise RuntimeError("已有 demo 实际内容与身份摘要不一致")
        if (existing_schema.equals(schema, check_metadata=True)
                and {key: value for key, value in existing_identity.items() if key != "generated_at_utc"} == expected_identity
                and existing_content_sha256 == expected_identity["output_content_sha256"]):
            preserved_existing = True
            installed_identity = existing_identity
        del existing_table
    if not preserved_existing:
        installed_identity = {**expected_identity, "generated_at_utc": datetime.now(timezone.utc).isoformat()}
        output_metadata = {**schema.metadata, b"demo_identity": json.dumps(installed_identity, ensure_ascii=False, allow_nan=False, sort_keys=True).encode()}
        staged_path = demo_path.with_name(demo_path.name + ".staging")
        if staged_path.exists():
            raise RuntimeError(f"已有未完成 staging，须先核查：{staged_path}")
        pq.write_table(output_table.replace_schema_metadata(output_metadata), staged_path,
            compression="zstd", compression_level=7, row_group_size=131072)
        staged_table = pq.read_table(staged_path)
        if not staged_table.schema.equals(output_table.replace_schema_metadata(output_metadata).schema, check_metadata=True):
            raise RuntimeError("staging Schema/metadata 回读不一致，保留 staging")
        if arrow_content_sha256(staged_table, schema, output_sort_keys) != expected_identity["output_content_sha256"]:
            raise RuntimeError("staging 实际数据摘要不一致，保留 staging")
        del staged_table
        os.replace(staged_path, demo_path)
    # 安装路径再次复读；不能仅凭写入成功或 metadata 中声明的摘要验收。
    installed_table = pq.read_table(demo_path)
    expected_metadata = {**schema.metadata, b"demo_identity": json.dumps(installed_identity, ensure_ascii=False, allow_nan=False, sort_keys=True).encode()}
    if not installed_table.schema.equals(schema.with_metadata(expected_metadata), check_metadata=True):
        raise RuntimeError("正式 demo Schema/metadata 回读不一致")
    if installed_table.num_rows != expected_identity["output_rows"] or arrow_content_sha256(installed_table, schema, output_sort_keys) != expected_identity["output_content_sha256"]:
        raise RuntimeError("正式 demo 实际数据摘要或行数不一致")
    return {"identity": installed_identity, "preserved_existing": preserved_existing, "file_bytes": demo_path.stat().st_size}


## 4. RB demo：读取主力映射

下面的单元格只读取上游 demo 和它的生成定义。物理字段、基础 metadata、生成代码摘要或实际内容摘要不匹配时抛出异常，后续分段单元格拒绝使用未核对的输入。实际内容摘要沿用上游口径：合并 Arrow chunks，恢复局部 Schema metadata，按现有行序写入 Arrow IPC stream 后计算 SHA-256。

In [7]:
demo_exchange_code = "XSGE"
demo_underlying_code = "RB"
demo_mapping_read_start_date = date(2015, 1, 1)
demo_mapping_read_end_date = date(2026, 9, 1)
demo_adjustment_start_date = date(2023, 10, 25)
demo_adjustment_end_date = date(2025, 10, 20)
demo_scope = {
    "mapping_read": {"start": demo_mapping_read_start_date.isoformat(), "end": demo_mapping_read_end_date.isoformat()},
    "minute_read_trading_dates": {"start": demo_adjustment_start_date.isoformat(), "end": demo_adjustment_end_date.isoformat()},
    "adjustment_trading_dates": {"start": demo_adjustment_start_date.isoformat(), "end": demo_adjustment_end_date.isoformat()},
    "timezone": ADJUSTMENT_TIMEZONE,
    "interval_closed": "both",
}
upstream_demo_path = candidate_root / "04_Research/a01_Methods/b01_MainContractSelection/b01_MainContractSelection_demo.parquet"
upstream_notebook_path = upstream_demo_path.with_name("b01_MainContractSelection.ipynb")
display(pd.DataFrame(demo_scope).T)


,start,end
mapping_read,2015-01-01,2026-09-01
minute_read_trading_dates,2023-10-25,2025-10-20
adjustment_trading_dates,2023-10-25,2025-10-20
timezone,Asia/Shanghai,Asia/Shanghai
interval_closed,both,both


In [8]:
# 清空核对标记，防止上次成功读取的变量掩盖本次失败。
verified_main_contract_daily_df = None
verified_upstream_identity = None
main_segment_daily_df = None
main_segment_summary_df = None
upstream_demo_table = pq.read_table(upstream_demo_path).combine_chunks()
upstream_metadata = upstream_demo_table.schema.metadata or {}
if not upstream_demo_table.schema.equals(MAIN_CONTRACT_DAILY_SCHEMA, check_metadata=False):
    raise RuntimeError("上游 demo 局部 Schema 不匹配，停止依赖计算")
if {key: value for key, value in upstream_metadata.items() if key != b"demo_identity"} != MAIN_CONTRACT_DAILY_SCHEMA.metadata:
    raise RuntimeError("上游 demo 基础 metadata 不匹配，停止依赖计算")
if b"demo_identity" not in upstream_metadata:
    raise RuntimeError("上游 demo 缺少 demo_identity，停止依赖计算")
upstream_demo_identity = json.loads(upstream_metadata[b"demo_identity"])
required_identity_keys = {
    "artifact_role", "producer_notebook", "method_version", "method_source_tag", "method_source_sha256",
    "input_identity", "output_rows", "output_content_sha256", "runtime", "generated_at_utc",
    "exchange_code", "underlying_code", "output_start_date", "output_end_date",
    "read_start_date", "read_end_date", "timezone", "interval_closed",
}
if not required_identity_keys.issubset(upstream_demo_identity):
    raise RuntimeError("上游 demo 生成身份字段不完整，停止依赖计算")
expected_producer = "04_Research/a01_Methods/b01_MainContractSelection/b01_MainContractSelection.ipynb"
expected_identity_values = {
    "artifact_role": "method_demo", "producer_notebook": expected_producer,
    "method_source_tag": "demo-dependency",
    "exchange_code": demo_exchange_code, "underlying_code": demo_underlying_code,
    "timezone": ADJUSTMENT_TIMEZONE, "interval_closed": "both",
}
for identity_key, expected_value in expected_identity_values.items():
    if upstream_demo_identity[identity_key] != expected_value:
        raise RuntimeError(f"上游生成身份 {identity_key} 不匹配，停止依赖计算")
if upstream_demo_identity["output_rows"] != upstream_demo_table.num_rows:
    raise RuntimeError("上游生成身份行数不匹配，停止依赖计算")
if not (
    date.fromisoformat(upstream_demo_identity["output_start_date"]) <= demo_mapping_read_start_date
    <= demo_adjustment_start_date <= demo_adjustment_end_date <= demo_mapping_read_end_date
    <= date.fromisoformat(upstream_demo_identity["output_end_date"])
):
    raise RuntimeError("读取或复权区间超出上游 demo 生成范围，停止依赖计算")

# 实际内容摘要与上游保存单元格使用完全相同的 Schema、行序和 IPC 口径。
upstream_content_table = upstream_demo_table.replace_schema_metadata(MAIN_CONTRACT_DAILY_SCHEMA.metadata)
upstream_content_buffer = pa.BufferOutputStream()
with pa.ipc.new_stream(upstream_content_buffer, upstream_content_table.schema) as upstream_content_writer:
    upstream_content_writer.write_table(upstream_content_table)
actual_upstream_content_sha256 = hashlib.sha256(upstream_content_buffer.getvalue().to_pybytes()).hexdigest()
if actual_upstream_content_sha256 != upstream_demo_identity["output_content_sha256"]:
    raise RuntimeError("上游实际内容摘要不匹配，停止依赖计算")

# 仅读取已明确标记的生成依赖；绘图定义不进入主力映射的数据身份。
upstream_notebook = json.loads(upstream_notebook_path.read_text(encoding="utf-8"))
upstream_dependency_cells = [
    cell for cell in upstream_notebook["cells"]
    if cell["cell_type"] == "code"
    and {"export", "demo-dependency"}.issubset(cell.get("metadata", {}).get("tags", []))
]
if not upstream_dependency_cells:
    raise RuntimeError("上游缺少 demo-dependency 定义，停止依赖计算")
upstream_dependency_source = "\n\n".join("".join(cell["source"]) for cell in upstream_dependency_cells)
actual_upstream_source_sha256 = hashlib.sha256(upstream_dependency_source.encode()).hexdigest()
if actual_upstream_source_sha256 != upstream_demo_identity["method_source_sha256"]:
    raise RuntimeError("上游生成代码摘要不匹配，请重算上游 demo 后再继续")

verified_main_contract_daily_df = upstream_content_table.to_pandas(types_mapper=pd.ArrowDtype)
verified_upstream_identity = upstream_demo_identity
print("局部 Schema、基础 metadata、生成代码、实际内容和读取范围核对通过。")
print(f"{len(verified_main_contract_daily_df)} 个交易日；{verified_main_contract_daily_df['trading_date'].min()} 至 {verified_main_contract_daily_df['trading_date'].max()}")
display(pd.DataFrame([
    {"field": field.name, "type": str(field.type), "nullable": field.nullable}
    for field in MAIN_CONTRACT_DAILY_SCHEMA
]))
display(pd.DataFrame([
    {"table": table_name, **input_identity}
    for table_name, input_identity in verified_upstream_identity["input_identity"].items()
]))
display(pd.DataFrame([{
    "producer": verified_upstream_identity["producer_notebook"],
    "generated_at_utc": verified_upstream_identity["generated_at_utc"],
    "method_version": verified_upstream_identity["method_version"],
    "source_sha256": actual_upstream_source_sha256,
    "content_sha256": actual_upstream_content_sha256,
}]))


局部 Schema、基础 metadata、生成代码、实际内容和读取范围核对通过。
2835 个交易日；2015-01-05 至 2026-09-01


,field,type,nullable
0,exchange_code,string,False
1,underlying_code,string,False
2,trading_date,date32[day],False
3,signal_trading_date,date32[day],True
4,incumbent_contract_code,string,True
5,incumbent_delist_date,date32[day],True
6,challenger_contract_code,string,True
7,challenger_delist_date,date32[day],True
8,main_contract_code,string,True
9,main_delist_date,date32[day],True


,table,rows,schema_version,sha256
0,dim_futures_bar_calendar,48558,1.4.0,f5f0554e3d8c032b3647dc50f25892c3a279175333fd4f...
1,dim_futures_contract_calendar,179727,1.2.0,f3ad9fddee9bd4c932ce112d467ccb8e796ac99bf5c5a0...
2,dim_futures_variety_calendar,4047,1.2.0,795f3d1e24d5d394dd0dd7a028f2f1747d24f86a4fb312...
3,fact_futures_daily,48558,1.5.0,febdc06361efadcac2a16d373fdd8b8906945ad422a767...


,producer,generated_at_utc,method_version,source_sha256,content_sha256
0,04_Research/a01_Methods/b01_MainContractSelect...,2026-10-04T19:41:13.566815+00:00,1.0.0,634d4d396f2ccd9ee2531fd912b03a79438ce8922a351d...,e4651db62b2d96ec67b7e1e6e3bd02720f3b4ad475d1f3...


## 5. 每日映射、换约与连续片段

展示上游状态和全部换约记录，再生成日级片段表及全部片段汇总。表格保存在内存，Notebook 保存本次展示结果。片段分钟边界由实际主力行情确定，并与日历期望网格核对；首尾分钟锚定原价，全部相邻实际分钟间隔参与均摊。

In [9]:
if verified_main_contract_daily_df is None or verified_upstream_identity is None:
    raise RuntimeError("上游身份未核对通过，不能生成片段")
mapping_scope_df = verified_main_contract_daily_df.loc[
    verified_main_contract_daily_df["trading_date"].between(demo_mapping_read_start_date, demo_mapping_read_end_date)
].copy()
display(mapping_scope_df.groupby(["selection_status", "selection_reason"], dropna=False).size().rename("trading_days").reset_index())
display(mapping_scope_df.head(8))
roll_records_df = mapping_scope_df.loc[mapping_scope_df["is_roll"].fillna(False)]
display(roll_records_df[["signal_trading_date", "trading_date", "incumbent_contract_code", "main_contract_code", "incumbent_signal_volume", "challenger_signal_volume", "selection_status"]])

main_segment_daily_df, main_segment_summary_df = build_main_contract_segments(
    mapping_scope_df,
    exchange_code=demo_exchange_code,
    underlying_code=demo_underlying_code,
    adjustment_start_date=demo_adjustment_start_date,
    adjustment_end_date=demo_adjustment_end_date,
)
for segment_df in (main_segment_daily_df, main_segment_summary_df):
    segment_df.attrs["scope"] = demo_scope
    segment_df.attrs["upstream_demo_identity"] = verified_upstream_identity
print(f"日级片段表：{len(main_segment_daily_df)} 行；连续主力片段：{len(main_segment_summary_df)} 个；未识别日：{main_segment_daily_df['segment_id'].isna().sum()}")
display(main_segment_daily_df.head(8))
display(main_segment_daily_df.tail(8))
display(main_segment_summary_df)


,selection_status,selection_reason,trading_days
0,held,threshold_not_exceeded,2801
1,rolled,volume_threshold_exceeded,34


,exchange_code,underlying_code,trading_date,signal_trading_date,incumbent_contract_code,incumbent_delist_date,challenger_contract_code,challenger_delist_date,main_contract_code,main_delist_date,incumbent_signal_volume,challenger_signal_volume,selected_signal_volume,candidate_count,unavailable_candidate_count,selection_status,selection_reason,is_roll,roll_trigger_ratio
0,XSGE,RB,2015-01-05,2014-12-31,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,4148900.0,288258.0,4148900.0,7,0,held,threshold_not_exceeded,False,1.1
1,XSGE,RB,2015-01-06,2015-01-05,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,2891694.0,159400.0,2891694.0,7,0,held,threshold_not_exceeded,False,1.1
2,XSGE,RB,2015-01-07,2015-01-06,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,7434538.0,381522.0,7434538.0,6,0,held,threshold_not_exceeded,False,1.1
3,XSGE,RB,2015-01-08,2015-01-07,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,4317868.0,244838.0,4317868.0,7,0,held,threshold_not_exceeded,False,1.1
4,XSGE,RB,2015-01-09,2015-01-08,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,3696668.0,191572.0,3696668.0,7,0,held,threshold_not_exceeded,False,1.1
5,XSGE,RB,2015-01-12,2015-01-09,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,3561070.0,226614.0,3561070.0,7,0,held,threshold_not_exceeded,False,1.1
6,XSGE,RB,2015-01-13,2015-01-12,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,5204600.0,322214.0,5204600.0,7,0,held,threshold_not_exceeded,False,1.1
7,XSGE,RB,2015-01-14,2015-01-13,RB1505.XSGE,2015-05-15,RB1510.XSGE,2015-10-15,RB1505.XSGE,2015-05-15,3337222.0,182992.0,3337222.0,7,0,held,threshold_not_exceeded,False,1.1


,signal_trading_date,trading_date,incumbent_contract_code,main_contract_code,incumbent_signal_volume,challenger_signal_volume,selection_status
37,2015-03-03,2015-03-04,RB1505.XSGE,RB1510.XSGE,1199714.0,1818088.0,rolled
129,2015-07-14,2015-07-15,RB1510.XSGE,RB1601.XSGE,1806312.0,2608720.0,rolled
202,2015-11-03,2015-11-04,RB1601.XSGE,RB1605.XSGE,1032118.0,1411662.0,rolled
289,2016-03-11,2016-03-14,RB1605.XSGE,RB1610.XSGE,4663174.0,5421854.0,rolled
398,2016-08-17,2016-08-18,RB1610.XSGE,RB1701.XSGE,3099566.0,3885986.0,rolled
464,2016-11-28,2016-11-29,RB1701.XSGE,RB1705.XSGE,2076172.0,2733408.0,rolled
538,2017-03-20,2017-03-21,RB1705.XSGE,RB1710.XSGE,1887018.0,2693626.0,rolled
632,2017-08-04,2017-08-07,RB1710.XSGE,RB1801.XSGE,3464504.0,4499920.0,rolled
691,2017-11-02,2017-11-03,RB1801.XSGE,RB1805.XSGE,2629562.0,3449374.0,rolled
790,2018-03-29,2018-03-30,RB1805.XSGE,RB1810.XSGE,2241628.0,3252386.0,rolled


日级片段表：481 行；连续主力片段：7 个；未识别日：0


,exchange_code,underlying_code,trading_date,signal_trading_date,main_contract_code,segment_id,segment_start_date,segment_end_date,selection_status,is_roll
0,XSGE,RB,2023-10-25,2023-10-24,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False
1,XSGE,RB,2023-10-26,2023-10-25,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False
2,XSGE,RB,2023-10-27,2023-10-26,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False
3,XSGE,RB,2023-10-30,2023-10-27,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False
4,XSGE,RB,2023-10-31,2023-10-30,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False
5,XSGE,RB,2023-11-01,2023-10-31,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False
6,XSGE,RB,2023-11-02,2023-11-01,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False
7,XSGE,RB,2023-11-03,2023-11-02,RB2401.XSGE,1,2023-10-25,2023-12-06,held,False


,exchange_code,underlying_code,trading_date,signal_trading_date,main_contract_code,segment_id,segment_start_date,segment_end_date,selection_status,is_roll
473,XSGE,RB,2025-10-09,2025-09-30,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False
474,XSGE,RB,2025-10-10,2025-10-09,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False
475,XSGE,RB,2025-10-13,2025-10-10,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False
476,XSGE,RB,2025-10-14,2025-10-13,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False
477,XSGE,RB,2025-10-15,2025-10-14,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False
478,XSGE,RB,2025-10-16,2025-10-15,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False
479,XSGE,RB,2025-10-17,2025-10-16,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False
480,XSGE,RB,2025-10-20,2025-10-17,RB2601.XSGE,7,2025-09-01,2025-10-20,held,False


,exchange_code,underlying_code,segment_id,main_contract_code,segment_start_date,segment_end_date,trading_day_count,is_final_segment
0,XSGE,RB,1,RB2401.XSGE,2023-10-25,2023-12-06,31,False
1,XSGE,RB,2,RB2405.XSGE,2023-12-07,2024-04-02,77,False
2,XSGE,RB,3,RB2410.XSGE,2024-04-03,2024-09-02,103,False
3,XSGE,RB,4,RB2501.XSGE,2024-09-03,2024-12-06,62,False
4,XSGE,RB,5,RB2505.XSGE,2024-12-09,2025-04-07,78,False
5,XSGE,RB,6,RB2510.XSGE,2025-04-08,2025-08-29,100,False
6,XSGE,RB,7,RB2601.XSGE,2025-09-01,2025-10-20,30,True


## 6. 日级分段验收与导入

核对日级输出覆盖本次复权区间，片段计数与起止日对应，并验证实际换约日开启新片段。重复合约及未识别日的分段行为在独立小样本检查中验证；真实 RB 结果是否出现这些情况由下方统计给出。

同名 `.py` 代理加载本页的 `export` 定义。调用方按根模板定位项目，并将 `04_Research` 加入模块路径后，可导入：

```python
from a01_Methods.b02_MainContinuousAdjustment.b02_MainContinuousAdjustment import (
    build_main_contract_segments, prepare_main_contract_minutes,
    adjust_main_contract_minutes, save_main_adjustment_demo,
)
```

导入只加载定义；读取、计算和保存均须显式调用。

In [10]:
expected_adjustment_df = mapping_scope_df.loc[
    mapping_scope_df["trading_date"].between(demo_adjustment_start_date, demo_adjustment_end_date)
]
assert main_segment_daily_df["trading_date"].tolist() == expected_adjustment_df["trading_date"].tolist()
assert len(main_segment_daily_df) == len(expected_adjustment_df)
identified_segments_df = main_segment_daily_df.loc[main_segment_daily_df["segment_id"].notna()]
assert int(main_segment_summary_df["trading_day_count"].sum()) == len(identified_segments_df)
assert main_segment_summary_df["segment_id"].tolist() == list(range(1, len(main_segment_summary_df) + 1))
assert identified_segments_df["trading_date"].between(
    identified_segments_df["segment_start_date"], identified_segments_df["segment_end_date"]
).all()
expected_segment_starts = identified_segments_df.loc[
    identified_segments_df["segment_id"].ne(identified_segments_df["segment_id"].shift()).fillna(True)
]
assert expected_segment_starts["trading_date"].tolist() == main_segment_summary_df["segment_start_date"].tolist()
if main_segment_daily_df["segment_id"].notna().all():
    assert len(main_segment_summary_df) == int(expected_adjustment_df["is_roll"].fillna(False).sum()) + 1
print("日级覆盖、连续片段编号、起止日与真实换约计数验收通过。")
print("重复担任主力的合约数：", int((main_segment_summary_df.groupby("main_contract_code").size() > 1).sum()))
if not main_segment_summary_df.empty:
    display(main_segment_summary_df.loc[main_segment_summary_df["is_final_segment"]])


日级覆盖、连续片段编号、起止日与真实换约计数验收通过。
重复担任主力的合约数： 0


,exchange_code,underlying_code,segment_id,main_contract_code,segment_start_date,segment_end_date,trading_day_count,is_final_segment
6,XSGE,RB,7,RB2601.XSGE,2025-09-01,2025-10-20,30,True


## 7. 只读分钟输入与覆盖

先读取指定品种和日期范围内的交易时段（Session）日历，再查看相关分钟文件的说明（metadata）并统计预计行数，最后读取该范围内的全合约分钟行情。覆盖情况采用正式日历的记录；`suspected_closed` 表示疑似休市，这些分钟仍记为缺失。下表列出日历要求采集的分钟均已齐备的连续交易日范围，并统计换约次数。

In [11]:
import pyarrow.dataset as ds
from config.settings import settings
from config.data_contracts import (
    FUTURES_BAR_CALENDAR_SCHEMA, validate_arrow_table, arrow_to_pandas,
)

# 年月表达式与交易日边界分开：前者裁剪 Hive 分区，后者包含完整夜盘交易日。
demo_month_starts = pd.date_range(
    pd.Timestamp(demo_adjustment_start_date).replace(day=1),
    pd.Timestamp(demo_adjustment_end_date).replace(day=1), freq="MS",
)
demo_month_filter = None
for month_start in demo_month_starts:
    month_filter = (ds.field("year") == month_start.year) & (ds.field("month") == month_start.month)
    demo_month_filter = month_filter if demo_month_filter is None else demo_month_filter | month_filter
scope_filter = (
    demo_month_filter & (ds.field("exchange_code") == demo_exchange_code)
    & (ds.field("underlying_code") == demo_underlying_code)
    & (ds.field("trading_date") >= demo_adjustment_start_date)
    & (ds.field("trading_date") <= demo_adjustment_end_date)
)
minute_partition_columns = FUTURES_MINUTE_SCHEMA.metadata[b"partition_columns"].decode().split(",")
bar_partition_columns = FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"partition_columns"].decode().split(",")
minute_dataset = ds.dataset(
    settings.futures_lake_root / "silver" / FUTURES_MINUTE_SCHEMA.metadata[b"table_name"].decode(),
    format="parquet", partitioning=ds.partitioning(
        pa.schema([FUTURES_MINUTE_SCHEMA.field(name) for name in minute_partition_columns]), flavor="hive"),
)
bar_calendar_dataset = ds.dataset(
    settings.futures_lake_root / "silver" / FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"table_name"].decode(),
    format="parquet", partitioning=ds.partitioning(
        pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(name) for name in bar_partition_columns]), flavor="hive"),
)
bar_calendar_filter = scope_filter & (ds.field("bar_frequency") == "1m")
# 在读取工作流现场核对物理 Schema，不以指定 dataset schema 遮蔽不兼容文件。
minute_input_identity = {}
for source_dataset, source_schema, source_filter, partition_columns in (
    (minute_dataset, FUTURES_MINUTE_SCHEMA, scope_filter, minute_partition_columns),
    (bar_calendar_dataset, FUTURES_BAR_CALENDAR_SCHEMA, bar_calendar_filter, bar_partition_columns),
):
    if source_dataset.schema.metadata != source_schema.metadata or set(source_dataset.schema.names) != set(source_schema.names):
        raise RuntimeError("silver Dataset 字段或表 metadata 不兼容")
    if any(not source_dataset.schema.field(name).equals(source_schema.field(name), check_metadata=True) for name in source_schema.names):
        raise RuntimeError("silver Dataset 字段类型或 metadata 不兼容")
    source_fragments = list(source_dataset.get_fragments(filter=source_filter))
    for fragment in source_fragments:
        physical_schema = fragment.physical_schema
        if physical_schema.metadata != source_schema.metadata:
            raise RuntimeError(f"文件 metadata 不兼容：{fragment.path}")
        if not (set(source_schema.names) - set(partition_columns) <= set(physical_schema.names) <= set(source_schema.names)):
            raise RuntimeError(f"文件物理字段不兼容：{fragment.path}")
        if any(not field.equals(source_schema.field(field.name), check_metadata=True) for field in physical_schema):
            raise RuntimeError(f"文件物理类型或字段 metadata 不兼容：{fragment.path}")
    minute_input_identity[source_schema.metadata[b"table_name"].decode()] = {
        "schema_version": source_schema.metadata[b"schema_version"].decode(),
        "partition_columns": partition_columns, "fragments": len(source_fragments),
        "selected_rows": source_dataset.count_rows(filter=source_filter),
        "trading_date_start": demo_adjustment_start_date.isoformat(),
        "trading_date_end": demo_adjustment_end_date.isoformat(),
        "timezone": ADJUSTMENT_TIMEZONE, "interval_closed": "both",
    }
print("读取前规模：", minute_input_identity)
bar_calendar_table = bar_calendar_dataset.to_table(filter=bar_calendar_filter, columns=FUTURES_BAR_CALENDAR_SCHEMA.names)
validate_arrow_table(bar_calendar_table, FUTURES_BAR_CALENDAR_SCHEMA)
bar_calendar_df = arrow_to_pandas(bar_calendar_table, FUTURES_BAR_CALENDAR_SCHEMA)
display(bar_calendar_df.groupby(["schedule_status", "is_fetch_required", "is_fetch_completed", "is_data_missing"], dropna=False).agg(
    session_count=("contract_code", "size"), expected_bars=("expected_bar_count", "sum"),
    actual_bars=("actual_bar_count", "sum"), missing_bars=("missing_bar_count", "sum"),
).reset_index())

# 信任已提交的覆盖状态，展示连续完整子区间，不为演示修改湖仓日历。
missing_calendar_dates = set(bar_calendar_df.loc[
    bar_calendar_df["is_fetch_required"] & bar_calendar_df["missing_bar_count"].gt(0), "trading_date"])
complete_spans = []
span_dates = []
for trading_date in main_segment_daily_df["trading_date"]:
    if trading_date in missing_calendar_dates:
        if span_dates:
            complete_spans.append(span_dates)
            span_dates = []
    else:
        span_dates.append(trading_date)
if span_dates:
    complete_spans.append(span_dates)
complete_span_records = []
for span in complete_spans:
    span_main_df = main_segment_daily_df.loc[main_segment_daily_df["trading_date"].isin(span)]
    roll_count = int(span_main_df["segment_id"].drop_duplicates().size - 1)
    complete_span_records.append({"start": span[0], "end": span[-1], "trading_days": len(span), "roll_count": roll_count})
complete_span_df = pd.DataFrame(complete_span_records)
display(complete_span_df)
if not complete_span_df.empty and complete_span_df["roll_count"].max() < 2:
    print("本次范围没有覆盖全部 required Session 且包含多次换约的连续子区间；保留固定原区间并显式携带缺失证据。")

futures_minute_table = minute_dataset.to_table(filter=scope_filter, columns=FUTURES_MINUTE_SCHEMA.names)
validate_arrow_table(futures_minute_table, FUTURES_MINUTE_SCHEMA)
futures_minute_df = arrow_to_pandas(futures_minute_table, FUTURES_MINUTE_SCHEMA)
assert len(futures_minute_df) == minute_input_identity["fact_futures_minute"]["selected_rows"]
print(f"实际读取 {len(futures_minute_df):,} 行，{futures_minute_df['contract_code'].nunique()} 合约，{futures_minute_df['trading_date'].nunique()} 交易日。")
print("实际时间戳：", futures_minute_df["bar_at"].min(), "至", futures_minute_df["bar_at"].max())
display(pd.DataFrame([{"field": field.name, "type": str(field.type), "nullable": field.nullable} for field in FUTURES_MINUTE_SCHEMA]))
display(futures_minute_df.head(6))


读取前规模： {'fact_futures_minute': {'schema_version': '1.5.0', 'partition_columns': ['exchange_code', 'underlying_code', 'year', 'month'], 'fragments': 26, 'selected_rows': 1972620, 'trading_date_start': '2023-10-25', 'trading_date_end': '2025-10-20', 'timezone': 'Asia/Shanghai', 'interval_closed': 'both'}, 'dim_futures_bar_calendar': {'schema_version': '1.4.0', 'partition_columns': ['bar_frequency', 'exchange_code', 'year', 'month'], 'fragments': 26, 'selected_rows': 23088, 'trading_date_start': '2023-10-25', 'trading_date_end': '2025-10-20', 'timezone': 'Asia/Shanghai', 'interval_closed': 'both'}}


,schedule_status,is_fetch_required,is_fetch_completed,is_data_missing,session_count,expected_bars,actual_bars,missing_bars
0,scheduled,True,True,False,22932,1972620,1972620,0
1,suspected_closed,True,True,True,156,18720,0,18720


,start,end,trading_days,roll_count
0,2023-10-25,2023-12-29,48,1
1,2024-01-03,2024-02-08,27,0
2,2024-02-20,2024-04-03,32,1
3,2024-04-09,2024-04-30,16,0
4,2024-05-07,2024-06-07,24,0
5,2024-06-12,2024-09-13,68,1
6,2024-09-19,2024-09-30,8,0
7,2024-10-09,2024-12-31,60,1
8,2025-01-03,2025-01-27,17,0
9,2025-02-06,2025-04-03,41,0


本次范围没有覆盖全部 required Session 且包含多次换约的连续子区间；保留固定原区间并显式携带缺失证据。


实际读取 1,972,620 行，36 合约，481 交易日。


实际时间戳： 2023-10-24 21:01:00+08:00 至 2025-10-20 15:00:00+08:00


,field,type,nullable
0,contract_code,string,False
1,exchange_code,string,False
2,underlying_code,string,False
3,trading_date,date32[day],False
4,session_number,int8,False
5,bar_at,"timestamp[us, tz=Asia/Shanghai]",False
6,open,double,True
7,high,double,True
8,low,double,True
9,close,double,True


,contract_code,exchange_code,underlying_code,trading_date,session_number,bar_at,open,high,low,close,volume,money,open_interest,source,updated_at,year,month
0,RB2311.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3600.0,3610.0,3600.0,3601.0,399.0,14370200.0,41870.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10
1,RB2311.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:02:00+08:00,3601.0,3604.0,3600.0,3602.0,108.0,3890270.0,41832.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10
2,RB2311.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:03:00+08:00,3601.0,3604.0,3601.0,3604.0,80.0,2882600.0,41829.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10
3,RB2311.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:04:00+08:00,3603.0,3603.0,3600.0,3602.0,280.0,10083590.0,41651.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10
4,RB2311.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:05:00+08:00,3600.0,3603.0,3600.0,3600.0,87.0,3133550.0,41635.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10
5,RB2311.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:06:00+08:00,3601.0,3602.0,3600.0,3601.0,86.0,3097380.0,41640.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10


## 8. 主力分钟片段与自然对数

`raw_*` 为原始 OHLC，`log_*` 为自然对数。数量字段沿用所选合约原值，不计算品种汇总。按日历定位主力缺失分钟；展示主力每日覆盖、片段分钟边界，以及真实换约两侧行情。范围首尾按交易日选择，夜盘时间戳可以早于范围起始自然日。

In [12]:
all_contract_log_minute_df, main_log_minute_df, preparation_diagnostics = prepare_main_contract_minutes(
    futures_minute_df, main_segment_daily_df, bar_calendar_df,
)
missing_main_minute_df = preparation_diagnostics["missing_main_minute_df"]
main_daily_coverage_df = preparation_diagnostics["main_daily_coverage_df"]
print("全合约 log 行数：", f"{len(all_contract_log_minute_df):,}", "主力 log 行数：", f"{len(main_log_minute_df):,}")
print("主力缺失分钟：", len(missing_main_minute_df), "未识别交易日：", len(preparation_diagnostics["unresolved_main_daily_df"]))
print("全合约中无同键主力分钟的时间点：", len(preparation_diagnostics["all_contract_times_without_main_df"]))
print("原价空值：", all_contract_log_minute_df[[f"raw_{name}" for name in OHLC_FIELDS]].isna().sum().to_dict())
print("log 空值：", all_contract_log_minute_df[[f"log_{name}" for name in OHLC_FIELDS]].isna().sum().to_dict())
display(main_daily_coverage_df.loc[main_daily_coverage_df["missing_minute_count"].gt(0)])
display(preparation_diagnostics["session_coverage_df"].loc[
    preparation_diagnostics["session_coverage_df"]["is_main"]
    & preparation_diagnostics["session_coverage_df"]["missing_bar_count"].gt(0),
    ["trading_date", "contract_code", "session_start_at", "session_end_at", "schedule_status", "expected_bar_count", "actual_bar_count", "quality_reason"],
])
display(main_log_minute_df.groupby("segment_id", sort=True).agg(
    contract_code=("contract_code", "first"), first_bar_at=("bar_at", "min"),
    last_bar_at=("bar_at", "max"), minute_count=("bar_at", "size"),
).reset_index())
display(all_contract_log_minute_df.head(8))
display(main_log_minute_df.head(4))
display(main_log_minute_df.tail(4))
raw_roll_sides_df = pd.concat([
    pd.concat([old_segment_df.tail(2), new_segment_df.head(2)]).assign(new_segment_id=new_id)
    for (old_id, old_segment_df), (new_id, new_segment_df) in zip(
        list(main_log_minute_df.groupby("segment_id", sort=True))[:-1],
        list(main_log_minute_df.groupby("segment_id", sort=True))[1:],
    )
], ignore_index=True)
display(raw_roll_sides_df[["new_segment_id", "bar_at", "trading_date", "contract_code", "segment_id", "raw_open", "raw_close", "log_open", "log_close", "volume", "money", "open_interest"]])

contract_daily_grid_df = preparation_diagnostics["session_coverage_df"].groupby(
    ["trading_date", "contract_code"], as_index=False,
).agg(expected_minute_count=("expected_bar_count", "sum"), observed_minute_count=("actual_bar_count", "sum"))
contract_daily_grid_df = contract_daily_grid_df.merge(
    main_daily_coverage_df[["trading_date", "observed_minute_count"]].rename(columns={"observed_minute_count": "main_observed_minute_count"}),
    on="trading_date", validate="many_to_one",
)
uneven_contract_grid_df = contract_daily_grid_df.loc[
    contract_daily_grid_df["observed_minute_count"].ne(contract_daily_grid_df["main_observed_minute_count"])
]
print("相对主力分钟条数不齐的合约交易日数：", len(uneven_contract_grid_df))
display(uneven_contract_grid_df.head(10))
assert main_log_minute_df["trading_date"].min() == demo_adjustment_start_date
assert main_log_minute_df["trading_date"].max() == demo_adjustment_end_date
assert int(bar_calendar_df["actual_bar_count"].sum()) == len(futures_minute_df)


全合约 log 行数： 1,972,620 主力 log 行数： 164,385
主力缺失分钟： 1560 未识别交易日： 0
全合约中无同键主力分钟的时间点： 0
原价空值： {'raw_open': 0, 'raw_high': 0, 'raw_low': 0, 'raw_close': 0}
log 空值： {'log_open': 0, 'log_high': 0, 'log_low': 0, 'log_close': 0}


,trading_date,main_contract_code,segment_id,expected_minute_count,observed_minute_count,missing_minute_count
48,2024-01-02,RB2405.XSGE,2,345,225,120
76,2024-02-19,RB2405.XSGE,2,345,225,120
109,2024-04-08,RB2410.XSGE,3,345,225,120
126,2024-05-06,RB2410.XSGE,3,345,225,120
151,2024-06-11,RB2410.XSGE,3,345,225,120
220,2024-09-18,RB2501.XSGE,4,345,225,120
229,2024-10-08,RB2501.XSGE,4,345,225,120
290,2025-01-02,RB2505.XSGE,5,345,225,120
308,2025-02-05,RB2505.XSGE,5,345,225,120
350,2025-04-07,RB2505.XSGE,5,345,225,120


,trading_date,contract_code,session_start_at,session_end_at,schedule_status,expected_bar_count,actual_bar_count,quality_reason
2608,2024-01-02,RB2405.XSGE,2023-12-29 21:00:00+08:00,2023-12-29 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
3528,2024-10-08,RB2501.XSGE,2024-09-30 21:00:00+08:00,2024-09-30 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
6460,2024-02-19,RB2405.XSGE,2024-02-08 21:00:00+08:00,2024-02-08 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
8464,2024-04-08,RB2410.XSGE,2024-04-03 21:00:00+08:00,2024-04-03 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
9328,2024-05-06,RB2410.XSGE,2024-04-30 21:00:00+08:00,2024-04-30 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
10224,2024-06-11,RB2410.XSGE,2024-06-07 21:00:00+08:00,2024-06-07 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
13320,2024-09-18,RB2501.XSGE,2024-09-13 21:00:00+08:00,2024-09-13 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
14176,2025-01-02,RB2505.XSGE,2024-12-31 21:00:00+08:00,2024-12-31 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
14868,2025-10-09,RB2601.XSGE,2025-09-30 21:00:00+08:00,2025-09-30 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。
15348,2025-02-05,RB2505.XSGE,2025-01-27 21:00:00+08:00,2025-01-27 23:00:00+08:00,suspected_closed,120,0,排除当前疑似 Session 后，其他完整 Session 重聚合与 JQData 日线一致；OHLC 容差为 tick_size/2，成交量和持仓量 绝对容差为 1e-6，成交额相对容差为 1e-6 且绝对容差为 0.01。该结论仅为 reconciled 旁证，不确认休市且不取消拉取。


,segment_id,contract_code,first_bar_at,last_bar_at,minute_count
0,1,RB2401.XSGE,2023-10-24 21:01:00+08:00,2023-12-06 15:00:00+08:00,10695
1,2,RB2405.XSGE,2023-12-06 21:01:00+08:00,2024-04-02 15:00:00+08:00,26325
2,3,RB2410.XSGE,2024-04-02 21:01:00+08:00,2024-09-02 15:00:00+08:00,35175
3,4,RB2501.XSGE,2024-09-02 21:01:00+08:00,2024-12-06 15:00:00+08:00,21150
4,5,RB2505.XSGE,2024-12-06 21:01:00+08:00,2025-04-07 15:00:00+08:00,26550
5,6,RB2510.XSGE,2025-04-07 21:01:00+08:00,2025-08-29 15:00:00+08:00,34260
6,7,RB2601.XSGE,2025-08-29 21:01:00+08:00,2025-10-20 15:00:00+08:00,10230


,contract_code,exchange_code,underlying_code,trading_date,session_number,bar_at,raw_open,raw_high,raw_low,raw_close,volume,money,open_interest,source,updated_at,year,month,log_open,log_high,log_low,log_close,main_contract_code,segment_id,is_main
0,RB2311.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3600.0,3610.0,3600.0,3601.0,399.0,14370200.0,41870.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.188689,8.191463,8.188689,8.188967,RB2401.XSGE,1,False
1,RB2312.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3621.0,3635.0,3621.0,3634.0,1083.0,39226470.0,35053.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.194506,8.198364,8.194506,8.198089,RB2401.XSGE,1,False
2,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3645.0,3649.0,3637.0,3637.0,57823.0,2106264520.0,1853183.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.201112,8.202208,8.198914,8.198914,RB2401.XSGE,1,True
3,RB2402.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3640.0,3640.0,3640.0,3640.0,2.0,72800.0,37064.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.199739,8.199739,8.199739,8.199739,RB2401.XSGE,1,False
4,RB2403.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3650.0,3662.0,3650.0,3654.0,2111.0,77090810.0,566815.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.202482,8.205765,8.202482,8.203578,RB2401.XSGE,1,False
5,RB2404.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3644.0,3644.0,3644.0,3644.0,1000.0,36440000.0,56496.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.200837,8.200837,8.200837,8.200837,RB2401.XSGE,1,False
6,RB2405.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3659.0,3660.0,3650.0,3650.0,1599.0,58459830.0,323008.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.204945,8.205218,8.202482,8.202482,RB2401.XSGE,1,False
7,RB2406.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3627.0,3627.0,3627.0,3627.0,0.0,0.0,63515.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.196161,8.196161,8.196161,8.196161,RB2401.XSGE,1,False


,contract_code,exchange_code,underlying_code,trading_date,session_number,bar_at,raw_open,raw_high,raw_low,raw_close,volume,money,open_interest,source,updated_at,year,month,log_open,log_high,log_low,log_close,main_contract_code,segment_id,is_main
0,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3645.0,3649.0,3637.0,3637.0,57823.0,2106264520.0,1853183.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.201112,8.202208,8.198914,8.198914,RB2401.XSGE,1,True
1,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:02:00+08:00,3638.0,3641.0,3638.0,3639.0,16299.0,593194040.0,1854606.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.199189,8.200014,8.199189,8.199464,RB2401.XSGE,1,True
2,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:03:00+08:00,3639.0,3642.0,3638.0,3642.0,14585.0,530931100.0,1856014.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.199464,8.200288,8.199189,8.200288,RB2401.XSGE,1,True
3,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:04:00+08:00,3641.0,3641.0,3638.0,3639.0,13086.0,476324740.0,1856001.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.200014,8.200014,8.199189,8.199464,RB2401.XSGE,1,True


,contract_code,exchange_code,underlying_code,trading_date,session_number,bar_at,raw_open,raw_high,raw_low,raw_close,volume,money,open_interest,source,updated_at,year,month,log_open,log_high,log_low,log_close,main_contract_code,segment_id,is_main
164381,RB2601.XSGE,XSGE,RB,2025-10-20,4,2025-10-20 14:57:00+08:00,3044.0,3044.0,3042.0,3043.0,2811.0,85533290.0,2008585.0,JQData_get_price_1m_skip_paused,2026-08-02 16:18:06+00:00,2025,10,8.020928,8.020928,8.02027,8.020599,RB2601.XSGE,7,True
164382,RB2601.XSGE,XSGE,RB,2025-10-20,4,2025-10-20 14:58:00+08:00,3043.0,3044.0,3042.0,3043.0,4136.0,125856370.0,2007227.0,JQData_get_price_1m_skip_paused,2026-08-02 16:18:06+00:00,2025,10,8.020599,8.020928,8.02027,8.020599,RB2601.XSGE,7,True
164383,RB2601.XSGE,XSGE,RB,2025-10-20,4,2025-10-20 14:59:00+08:00,3043.0,3044.0,3042.0,3044.0,6049.0,184087120.0,2006412.0,JQData_get_price_1m_skip_paused,2026-08-02 16:18:06+00:00,2025,10,8.020599,8.020928,8.02027,8.020928,RB2601.XSGE,7,True
164384,RB2601.XSGE,XSGE,RB,2025-10-20,4,2025-10-20 15:00:00+08:00,3044.0,3046.0,3044.0,3045.0,10260.0,312413310.0,2005926.0,JQData_get_price_1m_skip_paused,2026-08-02 16:18:06+00:00,2025,10,8.020928,8.021585,8.020928,8.021256,RB2601.XSGE,7,True


,new_segment_id,bar_at,trading_date,contract_code,segment_id,raw_open,raw_close,log_open,log_close,volume,money,open_interest
0,2,2023-12-06 14:59:00+08:00,2023-12-06,RB2401.XSGE,1,3917.0,3918.0,8.273081,8.273337,2729.0,106892620.0,593601.0
1,2,2023-12-06 15:00:00+08:00,2023-12-06,RB2401.XSGE,1,3918.0,3918.0,8.273337,8.273337,6249.0,244766880.0,592003.0
2,2,2023-12-06 21:01:00+08:00,2023-12-07,RB2405.XSGE,2,3953.0,3950.0,8.28223,8.281471,20234.0,800155600.0,1178241.0
3,2,2023-12-06 21:02:00+08:00,2023-12-07,RB2405.XSGE,2,3950.0,3951.0,8.281471,8.281724,6566.0,259440200.0,1179512.0
4,3,2024-04-02 14:59:00+08:00,2024-04-02,RB2405.XSGE,2,3469.0,3467.0,8.151622,8.151045,4612.0,159964520.0,1028988.0
5,3,2024-04-02 15:00:00+08:00,2024-04-02,RB2405.XSGE,2,3468.0,3463.0,8.151333,8.149891,5874.0,203597810.0,1026775.0
6,3,2024-04-02 21:01:00+08:00,2024-04-03,RB2410.XSGE,3,3500.0,3508.0,8.160518,8.162801,33958.0,1190691560.0,1658738.0
7,3,2024-04-02 21:02:00+08:00,2024-04-03,RB2410.XSGE,3,3508.0,3509.0,8.162801,8.163086,12479.0,437687540.0,1659649.0
8,4,2024-09-02 14:59:00+08:00,2024-09-02,RB2410.XSGE,3,3171.0,3177.0,8.061802,8.063693,5517.0,175037820.0,1083865.0
9,4,2024-09-02 15:00:00+08:00,2024-09-02,RB2410.XSGE,3,3177.0,3178.0,8.063693,8.064007,16198.0,516659380.0,1082665.0


相对主力分钟条数不齐的合约交易日数： 0


,trading_date,contract_code,expected_minute_count,observed_minute_count,main_observed_minute_count


## 9. 确定性小例子与边界行为

三天主力为 A → B → A，每天两分钟，验证重复合约仍分成三个片段。另检查单片段、单点和两点边界、缺失 log、同一分钟有两个非主力合约的无偏移情况、非正价格、重复键和锚点缺失。测试行情独立构造，不写入正式湖。

In [13]:
def make_small_adjustment_example(contract_sequence, minutes_per_day):
    """构造少量固定分钟，用于核对复权公式和输入边界。"""
    trading_dates = [date(2020, 1, 2), date(2020, 1, 3), date(2020, 1, 6)][:len(contract_sequence)]
    codes = ["RB2001.XSGE", "RB2005.XSGE"]
    daily_df = pd.DataFrame({
        "exchange_code": "XSGE", "underlying_code": "RB", "trading_date": trading_dates,
        "signal_trading_date": trading_dates, "main_contract_code": contract_sequence,
        "selection_status": "example", "is_roll": [False] + [True] * (len(trading_dates) - 1),
    })
    daily_df, summary_df = build_main_contract_segments(daily_df, exchange_code="XSGE", underlying_code="RB")
    rows, sessions = [], []
    first_log_open_by_day = [2.0, 3.0, 2.5]
    for day_index, trading_date in enumerate(trading_dates):
        session_start_at = pd.Timestamp(trading_date, tz=ADJUSTMENT_TIMEZONE) + pd.Timedelta(hours=9)
        for contract_index, contract_code in enumerate(codes):
            sessions.append({
                "bar_frequency": "1m", "contract_code": contract_code, "exchange_code": "XSGE",
                "underlying_code": "RB", "trading_date": trading_date, "session_number": 1,
                "session_start_at": session_start_at, "session_end_at": session_start_at + pd.Timedelta(minutes=minutes_per_day),
                "schedule_status": "example", "schedule_signal_reason": "synthetic",
                "is_fetch_required": True, "is_fetch_completed": True,
                "expected_bar_count": minutes_per_day, "actual_bar_count": minutes_per_day,
                "missing_bar_count": 0, "quality_status": "example", "quality_reason": "synthetic",
            })
            for minute_index in range(minutes_per_day):
                base_log = first_log_open_by_day[day_index] + minute_index * 0.1
                if contract_code != contract_sequence[day_index]:
                    base_log += 0.2
                rows.append({
                    "contract_code": contract_code, "exchange_code": "XSGE", "underlying_code": "RB",
                    "trading_date": trading_date, "session_number": 1,
                    "bar_at": session_start_at + pd.Timedelta(minutes=minute_index + 1),
                    "open": np.exp(base_log), "high": np.exp(base_log + .05),
                    "low": np.exp(base_log - .04), "close": np.exp(base_log + .02),
                    "volume": 100.0 + contract_index, "money": 10000.0 + minute_index,
                    "open_interest": 1000.0 + contract_index,
                    "source": "synthetic", "updated_at": pd.Timestamp("2020-01-10", tz="UTC"),
                    "year": 2020, "month": 1,
                })
    minute_df = pa.Table.from_pylist(rows, schema=FUTURES_MINUTE_SCHEMA).to_pandas(types_mapper=pd.ArrowDtype)
    return minute_df, daily_df, pd.DataFrame(sessions)

small_minute_df, small_daily_df, small_calendar_df = make_small_adjustment_example(
    ["RB2001.XSGE", "RB2005.XSGE", "RB2001.XSGE"], 2,
)
small_all_df, small_main_df, small_preparation = prepare_main_contract_minutes(small_minute_df, small_daily_df, small_calendar_df)
small_adjusted_main_df, small_adjusted_all_df, small_offset_df, small_adjustment = adjust_main_contract_minutes(
    small_main_df, small_all_df, small_daily_df, small_preparation["missing_main_minute_df"],
)
assert small_daily_df["segment_id"].tolist() == [1, 2, 3]
np.testing.assert_allclose(small_adjustment["roll_diagnostics_df"]["gap_log"], [.88, -.62], atol=1e-14)
np.testing.assert_allclose(small_offset_df["backward_log_shift"].to_numpy(dtype=float), [.26, .26, -.62, -.62, 0, 0], atol=1e-14)
np.testing.assert_allclose(small_offset_df["linear_log_deduction"].to_numpy(dtype=float), [.26, .208, .156, .104, .052, 0], atol=1e-14)
np.testing.assert_allclose(small_offset_df["log_adjustment"].to_numpy(dtype=float), [0, .052, -.776, -.724, -.052, 0], atol=1e-14)
# 每个相邻间隔都增加 .052；换约处先扣除原 gap，首尾 OHLC 精确保留。
small_roll_mask = small_main_df["segment_id"].to_numpy()[1:] != small_main_df["segment_id"].to_numpy()[:-1]
small_raw_steps = small_main_df["log_open"].to_numpy(dtype=float)[1:] - small_main_df["log_close"].to_numpy(dtype=float)[:-1]
small_adjusted_steps = small_adjusted_main_df["adjusted_log_open"].to_numpy(dtype=float)[1:] - small_adjusted_main_df["adjusted_log_close"].to_numpy(dtype=float)[:-1]
np.testing.assert_allclose(small_adjusted_steps, np.where(small_roll_mask, 0, small_raw_steps) + .052, atol=1e-14)
for name in OHLC_FIELDS:
    assert small_adjusted_main_df[f"adjusted_{name}"].iloc[[0, -1]].equals(small_main_df[f"raw_{name}"].iloc[[0, -1]])
display(small_offset_df)
display(small_adjustment["roll_diagnostics_df"])

for contracts, minutes, expected_intervals in ((["RB2001.XSGE"], 1, 0), (["RB2001.XSGE"], 3, 2), (["RB2001.XSGE", "RB2005.XSGE"], 1, 1)):
    test_minute_df, test_daily_df, test_calendar_df = make_small_adjustment_example(contracts, minutes)
    test_all_df, test_main_df, test_preparation = prepare_main_contract_minutes(test_minute_df, test_daily_df, test_calendar_df)
    test_adjusted_main_df, test_adjusted_all_df, test_offset_df, test_adjustment = adjust_main_contract_minutes(
        test_main_df, test_all_df, test_daily_df, test_preparation["missing_main_minute_df"],
    )
    assert test_adjustment["interval_count"] == expected_intervals
    assert test_offset_df["log_adjustment"].eq(0).all()
    if expected_intervals == 1:
        np.testing.assert_allclose(test_adjustment["roll_diagnostics_df"]["actual_residual_log"], [test_adjustment["total_log_gap"]], atol=1e-14)
        print("两点、一个间隔的分摊量：", test_adjustment["per_interval_log_gap"])

# 同一时点只有非主力合约行情：原价保留，调整结果为空，不借用相邻偏移。
test_minute_df, test_daily_df, test_calendar_df = make_small_adjustment_example(["RB2001.XSGE", "RB2005.XSGE"], 3)
interior_bar_at = test_minute_df["bar_at"].min() + pd.Timedelta(minutes=1)
partial_minute_df = test_minute_df.loc[~(test_minute_df["contract_code"].eq("RB2001.XSGE") & test_minute_df["bar_at"].eq(interior_bar_at))].copy()
# 非锚点 OHLC 空值保持为空。
partial_minute_df.loc[partial_minute_df["bar_at"].eq(interior_bar_at), "high"] = pd.NA
# 在同一无主力时点增加第二个非主力合约，原始 log 价差应有限。
extra_non_main_df = partial_minute_df.loc[partial_minute_df["bar_at"].eq(interior_bar_at)].copy()
extra_non_main_df["contract_code"] = pd.array(["RB2010.XSGE"], dtype=pd.ArrowDtype(pa.string()))
for name in OHLC_FIELDS:
    extra_non_main_df[name] = extra_non_main_df[name] * np.exp(.3)
partial_minute_df = pd.concat([partial_minute_df, extra_non_main_df], ignore_index=True)
partial_all_df, partial_main_df, partial_preparation = prepare_main_contract_minutes(partial_minute_df, test_daily_df, test_calendar_df)
partial_adjusted_main_df, partial_adjusted_all_df, partial_offset_df, partial_adjustment = adjust_main_contract_minutes(
    partial_main_df, partial_all_df, test_daily_df, partial_preparation["missing_main_minute_df"],
)
unsupported_rows_df = partial_adjusted_all_df.loc[partial_adjusted_all_df["bar_at"].eq(interior_bar_at)]
assert len(partial_preparation["missing_main_minute_df"]) == 1
assert len(unsupported_rows_df) == 2
assert unsupported_rows_df["log_close"].notna().all()
assert unsupported_rows_df["adjustment_status"].eq("no_main_offset").all()
assert unsupported_rows_df[[f"adjusted_{name}" for name in OHLC_FIELDS]].isna().all().all()
assert unsupported_rows_df[["backward_log_shift", "linear_log_deduction", "log_adjustment"]].isna().all().all()
assert partial_adjusted_all_df[ALL_CONTRACT_LOG_MINUTE_SCHEMA.names].equals(partial_all_df)
assert len(partial_adjusted_all_df) == len(partial_all_df)
partial_same_time = partial_adjusted_all_df["bar_at"].eq(partial_adjusted_all_df["bar_at"].shift()).fillna(False).to_numpy(dtype=bool)[1:]
partial_shift = partial_adjusted_all_df["log_adjustment"].to_numpy(dtype=float, na_value=np.nan)
for name in OHLC_FIELDS:
    original = partial_adjusted_all_df[f"log_{name}"].to_numpy(dtype=float, na_value=np.nan)
    adjusted = partial_adjusted_all_df[f"adjusted_log_{name}"].to_numpy(dtype=float, na_value=np.nan)
    valid_pairs = (partial_same_time & np.isfinite(partial_shift[:-1]) & np.isfinite(partial_shift[1:])
                   & np.isfinite(original[:-1]) & np.isfinite(original[1:])
                   & np.isfinite(adjusted[:-1]) & np.isfinite(adjusted[1:]))
    np.testing.assert_allclose(np.diff(adjusted)[valid_pairs], np.diff(original)[valid_pairs], atol=2e-14, rtol=1e-10)
assert unsupported_rows_df[[f"adjusted_log_{name}" for name in OHLC_FIELDS]].isna().all().all()
assert unsupported_rows_df["log_high"].isna().all()
assert len(partial_adjustment["all_contract_times_without_offset_df"]) == 1
for quantity in QUANTITY_FIELDS:
    assert partial_adjusted_all_df[quantity].equals(partial_all_df[quantity])

for bad_value in (0.0, -1.0, np.inf):
    bad_minute_df = small_minute_df.copy()
    bad_minute_df.loc[0, "open"] = bad_value
    try:
        prepare_main_contract_minutes(bad_minute_df, small_daily_df, small_calendar_df)
    except ValueError as error:
        assert "非正或非有限" in str(error)
    else:
        raise AssertionError("非正或非有限价格未被拒绝")
try:
    prepare_main_contract_minutes(pd.concat([small_minute_df, small_minute_df.iloc[:1]], ignore_index=True), small_daily_df, small_calendar_df)
except ValueError as error:
    assert "重复" in str(error)
else:
    raise AssertionError("重复键未被拒绝")

edge_minute_df = test_minute_df.loc[~(test_minute_df["contract_code"].eq("RB2001.XSGE") & test_minute_df["bar_at"].eq(test_minute_df["bar_at"].min()))]
edge_all_df, edge_main_df, edge_preparation = prepare_main_contract_minutes(edge_minute_df, test_daily_df, test_calendar_df)
try:
    adjust_main_contract_minutes(edge_main_df, edge_all_df, test_daily_df, edge_preparation["missing_main_minute_df"])
except ValueError as error:
    assert "边界缺失" in str(error)
else:
    raise AssertionError("片段边界缺失未被拒绝")
anchor_minute_df = small_minute_df.copy()
new_anchor_at = small_main_df.loc[small_main_df["segment_id"].eq(2), "bar_at"].min()
anchor_minute_df.loc[anchor_minute_df["bar_at"].eq(new_anchor_at) & anchor_minute_df["contract_code"].eq("RB2005.XSGE"), "open"] = pd.NA
anchor_all_df, anchor_main_df, anchor_preparation = prepare_main_contract_minutes(anchor_minute_df, small_daily_df, small_calendar_df)
try:
    adjust_main_contract_minutes(anchor_main_df, anchor_all_df, small_daily_df, anchor_preparation["missing_main_minute_df"])
except ValueError as error:
    assert "锚点缺失" in str(error)
else:
    raise AssertionError("换约空价格锚点未被拒绝")
print("确定性公式、重复合约分段、单点/两点、无偏移时点、空值、非法价格、重复键与边界拒绝检查通过。")

# 原价空值在具备偏移的时间点也保持为空；主力数量取各日选中合约。
assert small_main_df["volume"].tolist() == [100.0, 100.0, 101.0, 101.0, 100.0, 100.0]
null_minute_df = small_minute_df.copy()
null_minute_df.loc[0, "high"] = pd.NA
null_all_df, null_main_df, null_preparation = prepare_main_contract_minutes(null_minute_df, small_daily_df, small_calendar_df)
_, null_adjusted_all_df, _, _ = adjust_main_contract_minutes(null_main_df, null_all_df, small_daily_df, null_preparation["missing_main_minute_df"])
assert null_adjusted_all_df["raw_high"].isna().sum() == 1
assert null_adjusted_all_df["log_high"].isna().sum() == 1
assert null_adjusted_all_df["adjusted_log_high"].isna().sum() == 1
assert null_adjusted_all_df["adjusted_high"].isna().sum() == 1

# 两个不同主力合约拥有相同时间戳：即使合约主键不同也拒绝公共网格重叠。
overlap_minute_df, overlap_daily_df, overlap_calendar_df = make_small_adjustment_example(["RB2001.XSGE", "RB2005.XSGE"], 2)
overlap_minute_df = overlap_minute_df.merge(overlap_daily_df[["trading_date", "main_contract_code"]], on="trading_date", validate="many_to_one")
overlap_minute_df = overlap_minute_df.loc[overlap_minute_df["contract_code"].eq(overlap_minute_df["main_contract_code"])].drop(columns="main_contract_code")
second_day_mask = overlap_minute_df["trading_date"].eq(date(2020, 1, 3))
overlap_minute_df.loc[second_day_mask, "bar_at"] = overlap_minute_df.loc[second_day_mask, "bar_at"] - pd.Timedelta(days=1)
try:
    prepare_main_contract_minutes(overlap_minute_df, overlap_daily_df, overlap_calendar_df)
except ValueError as error:
    assert "时间重叠" in str(error)
else:
    raise AssertionError("主力时间重叠未被拒绝")

missing_segment_minute_df = test_minute_df.loc[~(test_minute_df["trading_date"].eq(date(2020, 1, 3)) & test_minute_df["contract_code"].eq("RB2005.XSGE"))]
missing_segment_all_df, missing_segment_main_df, missing_segment_preparation = prepare_main_contract_minutes(missing_segment_minute_df, test_daily_df, test_calendar_df)
try:
    adjust_main_contract_minutes(missing_segment_main_df, missing_segment_all_df, test_daily_df, missing_segment_preparation["missing_main_minute_df"])
except ValueError as error:
    assert "整段缺行情" in str(error)
else:
    raise AssertionError("整段缺行情未被拒绝")
unresolved_daily_df = test_daily_df.copy()
unresolved_daily_df.loc[1, ["main_contract_code", "segment_id"]] = pd.NA
unresolved_all_df, unresolved_main_df, unresolved_preparation = prepare_main_contract_minutes(test_minute_df, unresolved_daily_df, test_calendar_df)
assert len(unresolved_preparation["unresolved_main_daily_df"]) == 1
try:
    adjust_main_contract_minutes(unresolved_main_df, unresolved_all_df, unresolved_daily_df, unresolved_preparation["missing_main_minute_df"])
except ValueError as error:
    assert "每日主力全部识别" in str(error)
else:
    raise AssertionError("未识别交易日未被拒绝")
print("补充检查通过：有偏移处 null 保留、主力数量身份、时间重叠、整段缺行情和未识别日。")


,trading_date,bar_at,main_contract_code,segment_id,backward_log_shift,linear_log_deduction,log_adjustment
0,2020-01-02,2020-01-02 09:01:00+08:00,RB2001.XSGE,1,0.26,0.26,0.0
1,2020-01-02,2020-01-02 09:02:00+08:00,RB2001.XSGE,1,0.26,0.208,0.052
2,2020-01-03,2020-01-03 09:01:00+08:00,RB2005.XSGE,2,-0.62,0.156,-0.776
3,2020-01-03,2020-01-03 09:02:00+08:00,RB2005.XSGE,2,-0.62,0.104,-0.724
4,2020-01-06,2020-01-06 09:01:00+08:00,RB2001.XSGE,3,0.0,0.052,-0.052
5,2020-01-06,2020-01-06 09:02:00+08:00,RB2001.XSGE,3,0.0,0.0,0.0


,old_segment_id,new_segment_id,old_contract_code,new_contract_code,old_bar_at,new_bar_at,gap_log,old_linear_log_deduction,new_linear_log_deduction,expected_residual_log,actual_residual_log
0,1,2,RB2001.XSGE,RB2005.XSGE,2020-01-02 09:02:00+08:00,2020-01-03 09:01:00+08:00,0.88,0.208,0.156,0.052,0.052
1,2,3,RB2005.XSGE,RB2001.XSGE,2020-01-03 09:02:00+08:00,2020-01-06 09:01:00+08:00,-0.62,0.104,0.052,0.052,0.052


两点、一个间隔的分摊量： 0.98
确定性公式、重复合约分段、单点/两点、无偏移时点、空值、非法价格、重复键与边界拒绝检查通过。


补充检查通过：有偏移处 null 保留、主力数量身份、时间重叠、整段缺行情和未识别日。


## 10. RB 复权与数学核对

共同偏移从主力分钟计算，再广播到全合约。验收逐项检查锚点、同分钟 log 价差、全部 OHLC 同偏移、数量字段及换约残差。结果包含原价 `raw_*`、未调整自然对数 `log_*`、调整后自然对数 `adjusted_log_*` 和价格域 `adjusted_*`，下游直接选取相应价格域。

In [14]:
adjusted_main_minute_df, adjusted_all_contract_minute_df, minute_log_offset_df, adjustment_diagnostics = adjust_main_contract_minutes(
    main_log_minute_df, all_contract_log_minute_df, main_segment_daily_df, missing_main_minute_df,
)
print("复权主力行数：", f"{len(adjusted_main_minute_df):,}", "全合约行数：", f"{len(adjusted_all_contract_minute_df):,}")
print("总 log 缺口 G：", adjustment_diagnostics["total_log_gap"], "均摊间隔数 N-1：", adjustment_diagnostics["interval_count"])
print("每个间隔分摊的 log 差值：", adjustment_diagnostics["per_interval_log_gap"])
print("全合约无共同偏移时间点：", len(adjustment_diagnostics["all_contract_times_without_offset_df"]))
assert minute_log_offset_df["log_adjustment"].iloc[[0, -1]].eq(0).all()
np.testing.assert_allclose(np.diff(minute_log_offset_df["linear_log_deduction"].to_numpy(dtype=float)),
                           -adjustment_diagnostics["per_interval_log_gap"], atol=2e-14, rtol=1e-10)
for name in OHLC_FIELDS:
    assert adjusted_main_minute_df[f"adjusted_{name}"].iloc[[0, -1]].equals(main_log_minute_df[f"raw_{name}"].iloc[[0, -1]])
assert len(adjusted_all_contract_minute_df) == len(all_contract_log_minute_df)
assert len(adjusted_main_minute_df) == len(main_log_minute_df)
for quantity in QUANTITY_FIELDS:
    assert adjusted_all_contract_minute_df[quantity].equals(all_contract_log_minute_df[quantity])
    assert adjusted_main_minute_df[quantity].equals(main_log_minute_df[quantity])

broadcast_shift = adjusted_all_contract_minute_df["log_adjustment"].to_numpy(dtype=float, na_value=np.nan)
zero_shift_mask = broadcast_shift == 0
same_time_mask = (
    adjusted_all_contract_minute_df["bar_at"].eq(adjusted_all_contract_minute_df["bar_at"].shift())
    & adjusted_all_contract_minute_df["trading_date"].eq(adjusted_all_contract_minute_df["trading_date"].shift())
).fillna(False).to_numpy(dtype=bool)[1:]
max_log_spread_error = 0.0
max_ohlc_offset_error = 0.0
for name in OHLC_FIELDS:
    original_log = adjusted_all_contract_minute_df[f"log_{name}"].to_numpy(dtype=float, na_value=np.nan)
    adjusted_log = adjusted_all_contract_minute_df[f"adjusted_log_{name}"].to_numpy(dtype=float, na_value=np.nan)
    np.testing.assert_allclose(adjusted_log, original_log + broadcast_shift, atol=1e-14, rtol=1e-12, equal_nan=True)
    # 仅验收两侧价格和共同偏移均有效的同键合约对；无偏移另验收空值与原始字段保留。
    valid_spread_mask = (same_time_mask & np.isfinite(broadcast_shift[:-1]) & np.isfinite(broadcast_shift[1:])
                         & np.isfinite(original_log[:-1]) & np.isfinite(original_log[1:])
                         & np.isfinite(adjusted_log[:-1]) & np.isfinite(adjusted_log[1:]))
    spread_before = np.diff(original_log)[valid_spread_mask]
    spread_after = np.diff(adjusted_log)[valid_spread_mask]
    np.testing.assert_allclose(spread_after, spread_before, atol=2e-14, rtol=1e-10, equal_nan=True)
    if len(spread_before):
        max_log_spread_error = max(max_log_spread_error, float(np.nanmax(np.abs(spread_after - spread_before))))
    max_ohlc_offset_error = max(max_ohlc_offset_error, float(np.nanmax(np.abs((adjusted_log - original_log) - broadcast_shift))))
    assert adjusted_all_contract_minute_df.loc[zero_shift_mask, f"adjusted_{name}"].equals(
        adjusted_all_contract_minute_df.loc[zero_shift_mask, f"raw_{name}"])
roll_diagnostics_df = adjustment_diagnostics["roll_diagnostics_df"]
np.testing.assert_allclose(roll_diagnostics_df["actual_residual_log"], roll_diagnostics_df["expected_residual_log"], atol=2e-14, rtol=1e-10)
np.testing.assert_allclose(roll_diagnostics_df["actual_residual_log"], adjustment_diagnostics["per_interval_log_gap"], atol=2e-14, rtol=1e-10)
print("数学核对通过；最大同分钟 log 价差误差：", max_log_spread_error, "最大 OHLC 偏移误差：", max_ohlc_offset_error)
display(adjustment_diagnostics["segment_diagnostics_df"])
display(roll_diagnostics_df)
display(minute_log_offset_df.head(3))
display(minute_log_offset_df.tail(3))
first_real_roll = roll_diagnostics_df.iloc[0]
real_roll_sides_df = pd.concat([
    adjusted_main_minute_df.loc[adjusted_main_minute_df["segment_id"].eq(first_real_roll["old_segment_id"])].tail(3),
    adjusted_main_minute_df.loc[adjusted_main_minute_df["segment_id"].eq(first_real_roll["new_segment_id"])].head(3),
])
display(real_roll_sides_df[["bar_at", "trading_date", "contract_code", "segment_id", "raw_open", "raw_close", "log_adjustment", "adjusted_open", "adjusted_close", "volume", "money", "open_interest"]])
unsupported_mask = adjusted_all_contract_minute_df["adjustment_status"].eq("no_main_offset")
assert adjusted_all_contract_minute_df.loc[unsupported_mask, [
    *[f"adjusted_log_{name}" for name in OHLC_FIELDS], *[f"adjusted_{name}" for name in OHLC_FIELDS],
    "backward_log_shift", "linear_log_deduction", "log_adjustment",
]].isna().all().all()
assert adjusted_all_contract_minute_df[ALL_CONTRACT_LOG_MINUTE_SCHEMA.names].equals(all_contract_log_minute_df)
for name in ["contract_code", "main_contract_code", "segment_id", "is_main", "trading_date", "bar_at"]:
    assert adjusted_all_contract_minute_df[name].equals(all_contract_log_minute_df[name])
# 同一 A 保持四种 OHLC 的两两次序，包括原数据已有的异常。
from itertools import combinations
for left_name, right_name in combinations(OHLC_FIELDS, 2):
    before_difference = (adjusted_all_contract_minute_df[f"raw_{left_name}"] - adjusted_all_contract_minute_df[f"raw_{right_name}"]).to_numpy(dtype=float, na_value=np.nan)
    after_difference = (adjusted_all_contract_minute_df[f"adjusted_{left_name}"] - adjusted_all_contract_minute_df[f"adjusted_{right_name}"]).to_numpy(dtype=float, na_value=np.nan)
    valid_order = np.isfinite(before_difference) & np.isfinite(after_difference) & np.isfinite(broadcast_shift)
    assert np.array_equal(np.sign(before_difference[valid_order]), np.sign(after_difference[valid_order]))
raw_order_anomaly = (
    adjusted_all_contract_minute_df["raw_high"].lt(adjusted_all_contract_minute_df["raw_low"])
    | adjusted_all_contract_minute_df["raw_open"].gt(adjusted_all_contract_minute_df["raw_high"])
    | adjusted_all_contract_minute_df["raw_close"].gt(adjusted_all_contract_minute_df["raw_high"])
    | adjusted_all_contract_minute_df["raw_open"].lt(adjusted_all_contract_minute_df["raw_low"])
    | adjusted_all_contract_minute_df["raw_close"].lt(adjusted_all_contract_minute_df["raw_low"])
).fillna(False)
print("原始 OHLC 次序异常行数（原值保留）：", int(raw_order_anomaly.sum()))
real_order_anomaly_df = adjusted_all_contract_minute_df.loc[raw_order_anomaly]
display(real_order_anomaly_df[["contract_code", "bar_at", *[f"raw_{name}" for name in OHLC_FIELDS], *[f"adjusted_{name}" for name in OHLC_FIELDS]]].head(8))

# 独立截取首个真实换约两侧交易日，再调用一次准备与复权；不重新读取湖。
old_roll_trading_date = main_log_minute_df.loc[main_log_minute_df["bar_at"].eq(first_real_roll["old_bar_at"]), "trading_date"].iloc[0]
new_roll_trading_date = main_log_minute_df.loc[main_log_minute_df["bar_at"].eq(first_real_roll["new_bar_at"]), "trading_date"].iloc[0]
real_window_daily_df, _ = build_main_contract_segments(
    mapping_scope_df, exchange_code=demo_exchange_code, underlying_code=demo_underlying_code,
    adjustment_start_date=old_roll_trading_date, adjustment_end_date=new_roll_trading_date,
)
real_window_all_df, real_window_main_df, real_window_preparation = prepare_main_contract_minutes(
    futures_minute_df, real_window_daily_df, bar_calendar_df,
)
real_window_adjusted_main_df, real_window_adjusted_all_df, real_window_offset_df, real_window_adjustment = adjust_main_contract_minutes(
    real_window_main_df, real_window_all_df, real_window_daily_df, real_window_preparation["missing_main_minute_df"],
)
assert len(real_window_adjustment["roll_diagnostics_df"]) == 1
real_window_per_interval_gap = real_window_adjustment["total_log_gap"] / (len(real_window_main_df) - 1)
np.testing.assert_allclose(real_window_adjustment["roll_diagnostics_df"]["actual_residual_log"], real_window_per_interval_gap, atol=2e-14)
assert real_window_offset_df["log_adjustment"].iloc[[0, -1]].eq(0).all()
print("独立真实换约调用：", old_roll_trading_date, "至", new_roll_trading_date, "主力分钟", len(real_window_main_df))
display(real_window_adjustment["roll_diagnostics_df"])

# 展示两处完整区间下的局部换约；每一处都留下相同的间隔分摊量。
selected_real_rolls_df = roll_diagnostics_df.iloc[[0, -1]]
for roll_case in selected_real_rolls_df.itertuples(index=False):
    case_sides_df = pd.concat([
        adjusted_main_minute_df.loc[adjusted_main_minute_df["segment_id"].eq(roll_case.old_segment_id)].tail(3),
        adjusted_main_minute_df.loc[adjusted_main_minute_df["segment_id"].eq(roll_case.new_segment_id)].head(3),
    ])
    print("换约案例：", roll_case.old_contract_code, "→", roll_case.new_contract_code,
          "理论 log 残差", roll_case.expected_residual_log, "实际", roll_case.actual_residual_log)
    display(case_sides_df[["bar_at", "trading_date", "contract_code", "segment_id", "raw_open", "raw_close", "log_adjustment", "adjusted_log_open", "adjusted_log_close", "adjusted_open", "adjusted_close", "volume"]])


复权主力行数： 164,385 全合约行数： 1,972,620
总 log 缺口 G： 0.09055810936841446 均摊间隔数 N-1： 164384
每个间隔分摊的 log 差值： 5.508936962746646e-07
全合约无共同偏移时间点： 0


数学核对通过；最大同分钟 log 价差误差： 1.7763568394002505e-15 最大 OHLC 偏移误差： 8.881784197001252e-16


,segment_id,main_contract_code,first_bar_at,last_bar_at,first_trading_date,last_trading_date,minute_count,backward_log_shift,missing_minute_count,is_final_segment
0,1,RB2401.XSGE,2023-10-24 21:01:00+08:00,2023-12-06 15:00:00+08:00,2023-10-25,2023-12-06,10695,0.090558,0,False
1,2,RB2405.XSGE,2023-12-06 21:01:00+08:00,2024-04-02 15:00:00+08:00,2023-12-07,2024-04-02,26325,0.081665,240,False
2,3,RB2410.XSGE,2024-04-02 21:01:00+08:00,2024-09-02 15:00:00+08:00,2024-04-03,2024-09-02,35175,0.071037,360,False
3,4,RB2501.XSGE,2024-09-02 21:01:00+08:00,2024-12-06 15:00:00+08:00,2024-09-03,2024-12-06,21150,0.056976,240,False
4,5,RB2505.XSGE,2024-12-06 21:01:00+08:00,2025-04-07 15:00:00+08:00,2024-12-09,2025-04-07,26550,0.043267,360,False
5,6,RB2510.XSGE,2025-04-07 21:01:00+08:00,2025-08-29 15:00:00+08:00,2025-04-08,2025-08-29,34260,0.019231,240,False
6,7,RB2601.XSGE,2025-08-29 21:01:00+08:00,2025-10-20 15:00:00+08:00,2025-09-01,2025-10-20,10230,0.0,120,True


,old_segment_id,new_segment_id,old_contract_code,new_contract_code,old_bar_at,new_bar_at,gap_log,old_linear_log_deduction,new_linear_log_deduction,expected_residual_log,actual_residual_log
0,1,2,RB2401.XSGE,RB2405.XSGE,2023-12-06 15:00:00+08:00,2023-12-06 21:01:00+08:00,0.008893,0.084667,0.084666,5.508937e-07,5.508937e-07
1,2,3,RB2405.XSGE,RB2410.XSGE,2024-04-02 15:00:00+08:00,2024-04-02 21:01:00+08:00,0.010628,0.070165,0.070164,5.508937e-07,5.508937e-07
2,3,4,RB2410.XSGE,RB2501.XSGE,2024-09-02 15:00:00+08:00,2024-09-02 21:01:00+08:00,0.014061,0.050787,0.050786,5.508937e-07,5.508937e-07
3,4,5,RB2501.XSGE,RB2505.XSGE,2024-12-06 15:00:00+08:00,2024-12-06 21:01:00+08:00,0.013709,0.039135,0.039135,5.508937e-07,5.508937e-07
4,5,6,RB2505.XSGE,RB2510.XSGE,2025-04-07 15:00:00+08:00,2025-04-07 21:01:00+08:00,0.024036,0.024509,0.024509,5.508937e-07,5.508937e-07
5,6,7,RB2510.XSGE,RB2601.XSGE,2025-08-29 15:00:00+08:00,2025-08-29 21:01:00+08:00,0.019231,0.005636,0.005635,5.508937e-07,5.508937e-07


,trading_date,bar_at,main_contract_code,segment_id,backward_log_shift,linear_log_deduction,log_adjustment
0,2023-10-25,2023-10-24 21:01:00+08:00,RB2401.XSGE,1,0.090558,0.090558,0.0
1,2023-10-25,2023-10-24 21:02:00+08:00,RB2401.XSGE,1,0.090558,0.090558,0.000001
2,2023-10-25,2023-10-24 21:03:00+08:00,RB2401.XSGE,1,0.090558,0.090557,0.000001


...

,old_segment_id,new_segment_id,old_contract_code,new_contract_code,old_bar_at,new_bar_at,gap_log,old_linear_log_deduction,new_linear_log_deduction,expected_residual_log,actual_residual_log
0,1,2,RB2401.XSGE,RB2405.XSGE,2023-12-06 15:00:00+08:00,2023-12-06 21:01:00+08:00,0.008893,0.004453,0.00444,0.000013,0.000013


换约案例： RB2401.XSGE → RB2405.XSGE 理论 log 残差 5.508936962746646e-07 实际 5.508936951059695e-07


,bar_at,trading_date,contract_code,segment_id,raw_open,raw_close,log_adjustment,adjusted_log_open,adjusted_log_close,adjusted_open,adjusted_close,volume
10692,2023-12-06 14:58:00+08:00,2023-12-06,RB2401.XSGE,1,3915.0,3916.0,0.00589,8.278461,8.278716,3938.128005,3939.133913,4205.0
10693,2023-12-06 14:59:00+08:00,2023-12-06,RB2401.XSGE,1,3917.0,3918.0,0.005891,8.278972,8.279227,3940.141991,3941.147899,2729.0
10694,2023-12-06 15:00:00+08:00,2023-12-06,RB2401.XSGE,1,3918.0,3918.0,0.005891,8.279228,8.279228,3941.15007,3941.15007,6249.0
10695,2023-12-06 21:01:00+08:00,2023-12-07,RB2405.XSGE,2,3953.0,3950.0,-0.003002,8.279228,8.278469,3941.152241,3938.161233,20234.0
10696,2023-12-06 21:02:00+08:00,2023-12-07,RB2405.XSGE,2,3950.0,3951.0,-0.003001,8.27847,8.278723,3938.163402,3939.160406,6566.0
10697,2023-12-06 21:03:00+08:00,2023-12-07,RB2405.XSGE,2,3950.0,3948.0,-0.003001,8.27847,8.277964,3938.165572,3936.171564,4506.0


换约案例： RB2510.XSGE → RB2601.XSGE 理论 log 残差 5.508936962746646e-07 实际 5.508936951059695e-07


,bar_at,trading_date,contract_code,segment_id,raw_open,raw_close,log_adjustment,adjusted_log_open,adjusted_log_close,adjusted_open,adjusted_close,volume
154152,2025-08-29 14:58:00+08:00,2025-08-29,RB2510.XSGE,6,3089.0,3089.0,0.013595,8.049197,8.049197,3131.280516,3131.280516,3744.0
154153,2025-08-29 14:59:00+08:00,2025-08-29,RB2510.XSGE,6,3090.0,3090.0,0.013595,8.049522,8.049522,3132.295929,3132.295929,4279.0
154154,2025-08-29 15:00:00+08:00,2025-08-29,RB2510.XSGE,6,3089.0,3090.0,0.013596,8.049198,8.049522,3131.283966,3132.297655,9031.0
154155,2025-08-29 21:01:00+08:00,2025-09-01,RB2601.XSGE,7,3150.0,3157.0,-0.005635,8.049523,8.051742,3132.299381,3139.260046,37981.0
154156,2025-08-29 21:02:00+08:00,2025-09-01,RB2601.XSGE,7,3157.0,3154.0,-0.005635,8.051743,8.050792,3139.261775,3136.278631,14587.0
154157,2025-08-29 21:03:00+08:00,2025-09-01,RB2601.XSGE,7,3153.0,3150.0,-0.005634,8.050476,8.049524,3135.285977,3132.302832,11365.0


## 11. 结果身份、保存与复读

全合约分钟长表使用本方法局部 Arrow Schema。原价、自然对数、调整后自然对数、价格域结果、数量、主力身份及共同偏移同时保存。`is_main=True` 即为连续主力行情，`segment_id` 标识主力片段，`backward_log_shift`、`linear_log_deduction`、`log_adjustment` 可定位偏移。

metadata 记录固定范围、信息时区、复权和缺失口径、上游 demo 身份、实际分钟/Session 输入内容摘要、生成定义与局部 Schema、参数、运行环境、实际输出内容摘要。数据摘要按固定 Schema、明确排序、合并 chunks 后的 Arrow IPC 计算，排除 `demo_identity`，避免循环依赖。

写入 staging 后复读，再原子安装并复读正式文件。已有身份与实际内容均一致时保留原字节和生成时间。严格复现还需取得记录的输入与代码；摘要本身不保存原始输入。

In [15]:
import platform

method_notebook_path = candidate_root / "04_Research/a01_Methods/b02_MainContinuousAdjustment/b02_MainContinuousAdjustment.ipynb"
method_notebook = json.loads(method_notebook_path.read_text(encoding="utf-8"))
method_dependency_source = "\n\n".join("".join(cell["source"]) for cell in method_notebook["cells"]
    if cell["cell_type"] == "code" and {"export", "demo-dependency"}.issubset(cell.get("metadata", {}).get("tags", [])))
for input_table, input_schema in ((futures_minute_table, FUTURES_MINUTE_SCHEMA), (bar_calendar_table, FUTURES_BAR_CALENDAR_SCHEMA)):
    table_name = input_schema.metadata[b"table_name"].decode()
    input_sort_keys = [(name, "ascending") for name in input_schema.metadata[b"primary_key"].decode().split(",")]
    minute_input_identity[table_name]["content_sha256"] = arrow_content_sha256(input_table, input_schema, input_sort_keys)
    minute_input_identity[table_name]["schema_sha256"] = hashlib.sha256(input_schema.serialize().to_pybytes()).hexdigest()
    minute_input_identity[table_name]["content_sort_keys"] = input_sort_keys

adjustment_demo_identity = {
    "artifact_role": "method_demo",
    "producer_notebook": method_notebook_path.relative_to(candidate_root).as_posix(),
    "method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION,
    "method_source_tag": "demo-dependency",
    "method_source_sha256": hashlib.sha256(method_dependency_source.encode()).hexdigest(),
    "dependency_source_sha256": {
        "notebook_loader": hashlib.sha256((candidate_root / "04_Research/a00_notebook_loader.py").read_bytes()).hexdigest(),
        "data_contracts": hashlib.sha256((candidate_root / "config/data_contracts.py").read_bytes()).hexdigest(),
    },
    "exchange_code": demo_exchange_code, "underlying_code": demo_underlying_code,
    "scope": demo_scope, "timezone": ADJUSTMENT_TIMEZONE, "interval_closed": "both",
    "actual_bar_at_start": str(adjusted_all_contract_minute_df["bar_at"].min()),
    "actual_bar_at_end": str(adjusted_all_contract_minute_df["bar_at"].max()),
    "upstream_demo_identity": verified_upstream_identity,
    "input_identity": minute_input_identity,
    "parameters": {
        "price_transform": "natural_log_positive_finite; null stays null",
        "gap": "new segment first log_open - old segment last log_close",
        "backward_shift": "reverse cumulative future gaps; final segment zero",
        "linear_deduction": "linspace(G,0,N) on all observed main minutes; N-1 equal intervals including rolls; N=1:G=0",
        "endpoint_policy": "first and last observed main minute OHLC unchanged",
        "common_offset": "B_segment - linear_deduction; broadcast by trading_date,bar_at",
        "zero_offset_prices": "copy original exactly",
        "missing_policy": "report interior missing; reject missing segment edges/anchors/unresolved days; no filling",
        "no_main_offset": "adjusted fields null; original/log/quantities retained",
        "quantity_policy": "original per-contract; main uses selected contract; no variety aggregate",
    },
    "runtime": {"python": sys.version.split()[0], "python_executable": sys.executable,
        "pandas": pd.__version__, "pyarrow": pa.__version__, "numpy": np.__version__, "platform": platform.platform()},
    "diagnostic_summary": {
        "contracts": int(adjusted_all_contract_minute_df["contract_code"].nunique()),
        "trading_days": int(main_segment_daily_df["trading_date"].nunique()),
        "main_minutes": len(adjusted_main_minute_df), "segments": len(main_segment_summary_df),
        "rolls": len(roll_diagnostics_df), "missing_main_minutes": len(missing_main_minute_df),
        "missing_main_trading_dates": int(missing_main_minute_df["trading_date"].nunique()),
        "no_main_offset_times": len(adjustment_diagnostics["all_contract_times_without_offset_df"]),
        "original_ohlc_order_anomaly_rows": int(raw_order_anomaly.sum()),
        "total_log_gap": adjustment_diagnostics["total_log_gap"],
        "interval_count": adjustment_diagnostics["interval_count"],
        "per_interval_log_gap": adjustment_diagnostics["per_interval_log_gap"],
    },
}
adjustment_demo_path = method_notebook_path.with_name(method_notebook_path.stem + "_demo.parquet")
adjustment_demo_save = save_main_adjustment_demo(adjusted_all_contract_minute_df,
    demo_path=adjustment_demo_path, demo_identity=adjustment_demo_identity)
print("保存并复读验收：", adjustment_demo_path)
print("行数：", adjustment_demo_save["identity"]["output_rows"], "文件 MiB：", round(adjustment_demo_save["file_bytes"] / 2**20, 2),
    "保留已有字节：", adjustment_demo_save["preserved_existing"])
print("输出内容 SHA-256：", adjustment_demo_save["identity"]["output_content_sha256"])
display(pd.DataFrame([{"field": field.name, "type": str(field.type), "nullable": field.nullable} for field in ALL_CONTRACT_ADJUSTED_MINUTE_SCHEMA]))
# 从已保存长表明确选出主力，核对时间、片段、数量和价格；不重新运行主力识别。
saved_demo_table = pq.read_table(adjustment_demo_path)
saved_main_df = saved_demo_table.filter(saved_demo_table["is_main"]).to_pandas(types_mapper=pd.ArrowDtype)
assert saved_main_df.equals(adjusted_main_minute_df)
display(saved_main_df.head(3))
display(saved_main_df.tail(3))


保存并复读验收： E:\Latitude_Analytics_v2\04_Research\a01_Methods\b02_MainContinuousAdjustment\b02_MainContinuousAdjustment_demo.parquet
行数： 1972620 文件 MiB： 160.91 保留已有字节： False
输出内容 SHA-256： 40f6d189236d4ad21fcf9171ac0953a0d228b31b933df64f8cd2bc8c10c81170


,field,type,nullable
0,contract_code,string,False
1,exchange_code,string,False
2,underlying_code,string,False
3,trading_date,date32[day],False
4,session_number,int8,False
5,bar_at,"timestamp[us, tz=Asia/Shanghai]",False
6,raw_open,double,True
7,raw_high,double,True
8,raw_low,double,True
9,raw_close,double,True


,contract_code,exchange_code,underlying_code,trading_date,session_number,bar_at,raw_open,raw_high,raw_low,raw_close,volume,money,open_interest,source,updated_at,year,month,log_open,log_high,log_low,log_close,main_contract_code,segment_id,is_main,backward_log_shift,linear_log_deduction,log_adjustment,adjustment_status,adjusted_log_open,adjusted_log_high,adjusted_log_low,adjusted_log_close,adjusted_open,adjusted_high,adjusted_low,adjusted_close
0,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:01:00+08:00,3645.0,3649.0,3637.0,3637.0,57823.0,2106264520.0,1853183.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.201112,8.202208,8.198914,8.198914,RB2401.XSGE,1,True,0.090558,0.090558,0.0,adjusted,8.201112,8.202208,8.198914,8.198914,3645.0,3649.0,3637.0,3637.0
1,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:02:00+08:00,3638.0,3641.0,3638.0,3639.0,16299.0,593194040.0,1854606.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.199189,8.200014,8.199189,8.199464,RB2401.XSGE,1,True,0.090558,0.090558,0.000001,adjusted,8.19919,8.200014,8.19919,8.199465,3638.002004,3641.002006,3638.002004,3639.002005
2,RB2401.XSGE,XSGE,RB,2023-10-25,1,2023-10-24 21:03:00+08:00,3639.0,3642.0,3638.0,3642.0,14585.0,530931100.0,1856014.0,JQData_get_price_1m_skip_paused,2026-08-02 16:17:39+00:00,2023,10,8.199464,8.200288,8.199189,8.200288,RB2401.XSGE,1,True,0.090558,0.090557,0.000001,adjusted,8.199465,8.200289,8.19919,8.200289,3639.004009,3642.004013,3638.004008,3642.004013


,contract_code,exchange_code,underlying_code,trading_date,session_number,bar_at,raw_open,raw_high,raw_low,raw_close,volume,money,open_interest,source,updated_at,year,month,log_open,log_high,log_low,log_close,main_contract_code,segment_id,is_main,backward_log_shift,linear_log_deduction,log_adjustment,adjustment_status,adjusted_log_open,adjusted_log_high,adjusted_log_low,adjusted_log_close,adjusted_open,adjusted_high,adjusted_low,adjusted_close
164382,RB2601.XSGE,XSGE,RB,2025-10-20,4,2025-10-20 14:58:00+08:00,3043.0,3044.0,3042.0,3043.0,4136.0,125856370.0,2007227.0,JQData_get_price_1m_skip_paused,2026-08-02 16:18:06+00:00,2025,10,8.020599,8.020928,8.02027,8.020599,RB2601.XSGE,7,True,0.0,0.000001,-0.000001,adjusted,8.020598,8.020927,8.020269,8.020598,3042.996647,3043.996646,3041.996648,3042.996647
164383,RB2601.XSGE,XSGE,RB,2025-10-20,4,2025-10-20 14:59:00+08:00,3043.0,3044.0,3042.0,3044.0,6049.0,184087120.0,2006412.0,JQData_get_price_1m_skip_paused,2026-08-02 16:18:06+00:00,2025,10,8.020599,8.020928,8.02027,8.020928,RB2601.XSGE,7,True,0.0,0.000001,-0.000001,adjusted,8.020599,8.020927,8.02027,8.020927,3042.998324,3043.998323,3041.998324,3043.998323
164384,RB2601.XSGE,XSGE,RB,2025-10-20,4,2025-10-20 15:00:00+08:00,3044.0,3046.0,3044.0,3045.0,10260.0,312413310.0,2005926.0,JQData_get_price_1m_skip_paused,2026-08-02 16:18:06+00:00,2025,10,8.020928,8.021585,8.020928,8.021256,RB2601.XSGE,7,True,0.0,0.0,0.0,adjusted,8.020928,8.021585,8.020928,8.021256,3044.0,3046.0,3044.0,3045.0
